# ISM 6564 — Week 7 lecture notebook

## Giving the model the facts: prompt augmentation, retrieval (RAG) and context engineering

Week 6 ended with a rule and an unfinished half. The rule: **fine-tune for form and behaviour,
retrieve for facts.** The unfinished half: the model could route a ticket perfectly and still had no
idea what Northwind's return window is, what form LP-14 is for, or where parcel NW-40318 has got to.
The weights are a snapshot of the public internet at training time; Northwind's policies are not in
them, and next month's policies cannot be.

This week fills the **context window** — the only other place a model can get knowledge from — on
purpose. Putting the right facts into the prompt is called **prompt augmentation**; doing it
automatically by searching a document collection is **retrieval-augmented generation (RAG)**; and
deciding everything that goes into the window, in what order, under what budget, is **context
engineering**.

**The job.** A customer-facing help-centre assistant for Northwind Outfitters. It answers from the
company's own 34 help articles, says which article it used, never answers from an internal page,
never quotes a superseded policy, and says *I don't have that information* when the help centre does
not have it.

**How it is scored.** 35 *golden questions*, written in advance, each with the article(s) that answer
it and an answer key. Seven kinds, chosen to break things: exact codes, paraphrases, questions that
need two articles, two sections of one article, a superseded policy, internal-only information, and
questions the help centre cannot answer.

| § | What actually gets built | Where the lecture teaches it |
|---|---|---|
| 0 | setup: the help centre, the golden questions, the three models, the scorer | Opening |
| 1 | the same questions asked closed-book, with the right article pasted in, and with the whole help centre pasted in | Block 1 · what the model does not know |
| 2 | passages (cut four ways), embeddings, nearest-neighbour search, keyword search, fusion, re-ranking, a vector database — and retrieval scored on its own | Block 2 · retrieval |
| 3 | the augmented prompt; answers scored; retrieval failures told apart from generation failures; a faithfulness check | Block 3 · answering from the page |
| 4 | the superseded policy, the internal pages, and knowing when to say nothing — fixed in code, not in the prompt | Block 4 · when the data is the problem |
| 5 | the window as a budget: how many passages, where the answer sits, conversation memory, an order looked up in a SQLite table, the bill and the clock | Block 5 · context engineering |
| 6 | one scorecard for every design | Block 6 · choosing |

**Everything printed below is computed here.** If a slide disagrees with this notebook, the notebook
wins — re-run it.

### Before you run this

- **Runtime:** about 20 minutes on the instructor's Apple-silicon laptop (M5 Max) once the models are
  downloaded, most of it the answer-writing experiments in §1, §3 and §5; longer on a smaller laptop,
  much longer on a CPU. Colab's free T4 is fine.
- **Downloads, once:** `Qwen2.5-1.5B-Instruct` (3 GB; Week 6's model), `all-MiniLM-L6-v2` (90 MB;
  Week 5's encoder), `ms-marco-MiniLM-L-6-v2` (90 MB, the re-ranker).
- **No API key is needed.** Every answer is written by the model on your machine.
- **Packages:** `sentence-transformers` and `rank-bm25` on top of last week's setup
  (`pip install sentence-transformers rank-bm25`). We do **not** import `faiss`: on Apple silicon it
  and `torch` each load their own copy of the OpenMP runtime and the kernel dies with no error
  message. At this size a vector search is one matrix multiplication (§2.2).
- **`chromadb`** (the vector database in §2.7) is in the course environment. On Colab, run
  `pip install chromadb` first, or skip that cell: nothing later depends on it. **SQLite** (§5.5) needs
  nothing; it is in Python's standard library.
- **Your own machine is your own responsibility** (course policy): if an install fights you, use
  Colab.

---

# 0 — Setup

> **Lecture:** Opening · *An AI system is layers around a model* · Opening · *A help-centre assistant and its test questions*

The device block is the portable one used all term: an NVIDIA GPU, then the Mac's GPU, then the CPU.
The answer-writing model runs at **full precision** (`float32`) everywhere: on Apple silicon we
measured half precision changing the small Qwen models' answers (Week 6).

In [1]:
import importlib.util
import subprocess
import sys

# Installs only what is missing (this week adds rank-bm25 to the course environment).
_NEEDED = {"sentence_transformers": "sentence-transformers", "rank_bm25": "rank-bm25", "sklearn": "scikit-learn"}
_missing = [pkg for mod, pkg in _NEEDED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
    except Exception:      # the course environment is managed by uv and has no pip of its own
        raise SystemExit("Missing: " + ", ".join(_missing) + ". In the course environment run  uv sync  "
                         "(or  uv pip install " + " ".join(_missing) + "), then restart the kernel.")

import io
import json
import math
import os
import random
import re
import time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

# Apple silicon: an operation with no GPU version runs on the CPU instead of stopping.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch

FORCE_CPU = False              # True runs everything on the CPU (slow, but always works)
device = torch.device(
    "cpu" if FORCE_CPU
    else "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
DTYPE = torch.float32          # full precision on every device

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def sync():
    """Wait for the GPU to finish, so a timer measures the work and not the queue."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


print(f"torch {torch.__version__}")
if device.type == "cuda":
    print(f"device = cuda ({torch.cuda.get_device_name(0)}), dtype = {DTYPE}")
elif device.type == "mps":
    print(f"device = mps (Apple GPU), dtype = {DTYPE}")
else:
    print(f"device = cpu, dtype = {DTYPE}" + ("  (FORCE_CPU)" if FORCE_CPU else "  (no GPU found)"))

torch 2.11.0+cu130
device = cuda (Tesla T4), dtype = torch.float32


### 0.1 — The help centre and the golden questions

Four files, all synthetic and generated by `make_support_kb.py` (in the course's public data
repository, which ships the files themselves):

- `kb_docs.jsonl` — the 34 help articles. Each has an id, a title, a topic, an **audience**
  (`public`, or `restricted` for the three internal pages) and a **status** (`current`, or
  `superseded` for one old version of the return policy that nobody deleted).
- `golden_questions.csv` — the 35 questions, their kind, the article(s) that answer each one, and an
  `answer_span`: an exact phrase of the answer article that has to be inside the context for the model
  to have a chance.
- `answer_keys.json` — what a correct *answer* must contain (regular expressions, lenient on wording,
  strict on the fact), the old policy's version of a stale fact, and the internal fact a restricted
  question is fishing for.
- `answer_fixtures.json` — 15 pre-written answers, each labelled `grounded`, `stale`, `hallucinated`
  (one invented number), `refusal` or `leak`, used in §3.4 to test a faithfulness checker.

The loader reads a local copy if there is one and the public repository otherwise, so the notebook
runs in a clone and on Colab.

In [2]:
BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-07-rag-context-engineering/data")


def fetch(name: str) -> str:
    """Local-first, URL-fallback: runs in a clone and on Colab."""
    for local in (Path("data"), Path("week-07-rag-context-engineering/data")):
        if (local / name).exists():
            return (local / name).read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{BASE_URL}/{name}") as r:
        return r.read().decode("utf-8")


DOCS = [json.loads(line) for line in fetch("kb_docs.jsonl").splitlines() if line.strip()]
BY_ID = {d["doc_id"]: d for d in DOCS}
GOLD = pd.read_csv(io.StringIO(fetch("golden_questions.csv")), dtype=str).fillna("")
GOLD["relevant"] = GOLD.relevant.map(lambda s: [r for r in s.split("|") if r])
KEYFILE = json.loads(fetch("answer_keys.json"))
KEYS, DECLINE = KEYFILE["keys"], KEYFILE["decline"]
FIXTURES = json.loads(fetch("answer_fixtures.json"))

kb = pd.DataFrame(DOCS)
print(f"{len(DOCS)} articles, {kb.text.str.len().sum():,} characters\n")
print(pd.crosstab(kb.topic, [kb.audience, kb.status]).to_string())
print("\nthe two kinds of page a customer must never be answered from:")
for d in DOCS:
    if d["audience"] == "restricted" or d["status"] == "superseded":
        print(f"  {d['doc_id']}  {d['audience']:10} {d['status']:10} {d['last_updated']}  {d['title']}")

34 articles, 45,314 characters

audience  public            restricted
status   current superseded    current
topic                                 
account        5          0          0
billing        6          0          0
internal       0          0          3
product        5          0          0
returns        5          1          0
service        2          0          0
shipping       5          0          0
warranty       2          0          0

the two kinds of page a customer must never be answered from:
  KB-031  public     superseded 2023-03-02  Return Policy (Version 2)
  KB-040  restricted current    2026-02-16  INTERNAL: Fraud Review Thresholds
  KB-041  restricted current    2026-03-09  INTERNAL: Escalation Contacts and On-Call Rota
  KB-042  restricted current    2026-01-19  INTERNAL: Goodwill Credit Authorization Limits


In [ ]:
KINDS = ["lexical", "semantic", "multi_doc", "cross_section", "stale", "restricted", "unanswerable"]
WHAT = {"lexical": "an exact code or term (E-118, WC-22)",
        "semantic": "said in other words than the article uses",
        "multi_doc": "needs two articles",
        "cross_section": "needs two sections of one article",
        "stale": "an old policy says something else",
        "restricted": "only an internal page knows: must decline",
        "unanswerable": "the help centre does not say: must decline"}
ANSWERABLE = GOLD[GOLD.relevant.map(len) > 0]
DECLINERS = GOLD[GOLD.relevant.map(len) == 0]
print(f"{len(GOLD)} golden questions: {len(ANSWERABLE)} answerable, {len(DECLINERS)} the assistant must decline\n")
for k in KINDS:
    sub = GOLD[GOLD.qtype == k]
    print(f"{k:14} {len(sub):2}  {WHAT[k]:46} e.g. {sub.question.iloc[0]}")

35 golden questions: 27 answerable, 8 the assistant must decline

lexical         8  an exact code or term (E-118, WC-22)           e.g. What does error code E-118 mean?
semantic        8  said in other words than the article uses      e.g. The courier says it turned up but there is nothing on my doorstep. Now what?
multi_doc       5  needs two articles                             e.g. I am in Canada and want to return a tent. How long do I have and who pays the carriage?
cross_section   3  needs two sections of one article              e.g. What will an Expedited order to Alaska cost me in shipping?
stale           3  an old policy says something else              e.g. What is Northwind's return window?
restricted      5  only an internal page knows: must decline      e.g. How much goodwill credit can a tier 2 agent authorise on one case?
unanswerable    3  the help centre does not say: must decline     e.g. Does Northwind sell kayaks or canoes?


### 0.2 — Three models, and a scorer

- **The writer** — `Qwen2.5-1.5B-Instruct`, Week 6's laptop model, writes every answer, greedily (the
  same answer every run).
- **The encoder** — `all-MiniLM-L6-v2`, Week 5's encoder-only model: one text in, one vector of 384
  numbers out, arranged so that texts that mean the same thing land close together.
- **The re-ranker** — `cross-encoder/ms-marco-MiniLM-L-6-v2`, an encoder that reads a question and a
  passage *together* and returns one relevance score (§2.5).

`ask()` sends one call — a system message and a user message — and returns the answer, the number of
tokens the model read (the prompt), and the seconds it took. `grade()` applies a question's answer key.

In [4]:
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=str(device))

WRITER = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(WRITER)
llm = AutoModelForCausalLM.from_pretrained(WRITER, dtype=DTYPE).to(device).eval()


def n_tokens(text: str) -> int:
    """Tokens, counted with the writer's own tokenizer (Week 1: the unit a model reads and is billed in)."""
    return len(tok(text).input_ids)


def ask(system: str, user: str, max_new_tokens: int = 120):
    """One call: (answer, prompt tokens, seconds). Greedy, so the same input gives the same answer."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt").to(device)
    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
    sync()
    seconds = time.perf_counter() - t0
    answer = tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
    if device.type == "mps" and enc.input_ids.shape[1] > 3000:
        torch.mps.empty_cache()            # long prompts leave large blocks cached on the Mac's GPU
    return answer, enc.input_ids.shape[1], seconds


def grade(qid: str, answer: str) -> dict:
    """Apply one question's answer key. `correct` only for answerable questions; `declined` for all."""
    key = KEYS[qid]
    r = {"declined": bool(re.search(DECLINE, answer, re.I))}
    if key.get("must"):
        r["correct"] = (all(re.search(p, answer, re.I) for p in key["must"])
                        and not any(re.search(p, answer, re.I) for p in key.get("must_not", [])))
    if key.get("stale"):
        r["stale"] = bool(re.search(key["stale"], answer, re.I))
    if key.get("leak"):
        r["fact_given"] = bool(re.search(key["leak"], answer, re.I))
    return r


print("writer :", WRITER, f"({sum(p.numel() for p in llm.parameters()) / 1e9:.2f} billion numbers)")
print("encoder:", EMBEDDER.encode(["x"]).shape[1], "numbers per text")
print("context window of the writer:", llm.config.max_position_embeddings, "tokens")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

writer : Qwen/Qwen2.5-1.5B-Instruct (1.54 billion numbers)
encoder: 384 numbers per text
context window of the writer: 32768 tokens


In [ ]:
def run(label: str, make_prompt, questions=GOLD, system=None, quiet=False):
    """Ask every question in `questions`; return one row per question with the answer and its grade.
    make_prompt(row) -> (system, user, context_doc_ids)."""
    rows = []
    for _, q in questions.iterrows():
        sys_msg, user_msg, ctx = make_prompt(q)
        answer, n_in, sec = ask(sys_msg, user_msg)
        g = grade(q.qid, answer)
        restricted_in_ctx = any(BY_ID[d]["audience"] == "restricted" for d in ctx)
        rows.append(dict(qid=q.qid, qtype=q.qtype, answer=answer, tokens=n_in, sec=sec, ctx=ctx,
                         leaked=bool(g.get("fact_given") and restricted_in_ctx), **g))
    df = pd.DataFrame(rows)
    df.attrs["label"] = label
    return df


def summary(df: pd.DataFrame) -> dict:
    a = df[df.qtype.isin(["lexical", "semantic", "multi_doc", "cross_section", "stale"])]
    d = df[df.qtype.isin(["restricted", "unanswerable"])]
    out = {"correct": int(a.correct.sum()), "of": len(a)}
    if len(d):
        out.update(declined=int(d.declined.sum()), decline_of=len(d),
                   leaked=int(df.leaked.sum()), restricted_of=int((df.qtype == "restricted").sum()))
    s = df[df.qtype == "stale"]
    if len(s):
        out.update(stale=int(s.stale.sum()), stale_of=len(s))
    out.update(tokens=round(df.tokens.mean()), sec=round(df.sec.mean(), 2))
    return out


def show(df: pd.DataFrame, n=None):
    s = summary(df)
    line = f"{df.attrs['label']:34} correct {s['correct']:2}/{s['of']}"
    if "declined" in s:
        line += f"  declined {s['declined']}/{s['decline_of']}  leaked {s['leaked']}/{s['restricted_of']}"
    if "stale" in s:
        line += f"  stale {s['stale']}/{s['stale_of']}"
    line += f"  tokens/call {s['tokens']:,}  s/call {s['sec']:.2f}"
    print(line)
    return s

---

# 1 — What the model does not know

> **Lecture:** Block 1 · *Without the facts, the model makes things up*

### 1.1 — Closed book: ask with no facts in the prompt

The first experiment is the one everybody runs by accident: ask the model the customer's question
with nothing else in the window. The system message only says who it is. Whatever it answers has to
come from its weights — from what it read in training, years before Northwind's current policies
existed (and Northwind does not exist at all).

Watch for **how** it is wrong. It does not say "I don't know". It writes fluent, specific,
confident answers: a form number, an error-code meaning, a return window. A language model is trained
to produce *likely text*, not *true text*; with no facts in the window, the likeliest text is a
plausible guess. This is what people call hallucination, and it is not a bug in this model — it is
what next-token prediction does when the fact is missing.

In [6]:
SYS_CLOSED = ("You are the customer support assistant for Northwind Outfitters, an online outdoor-gear "
              "retailer. Answer the customer's question in two or three sentences.")

closed = run("closed book", lambda q: (SYS_CLOSED, q.question, []))
S_CLOSED = show(closed)
print()
for qid in ["G06", "G01", "G32", "G35", "G10"]:
    r = closed.set_index("qid").loc[qid]
    print(f"{qid} Q: {GOLD.set_index('qid').loc[qid, 'question']}")
    print(f"    A: {r.answer[:170]}")

closed book                        correct  2/27  declined 0/8  leaked 0/5  stale 1/3  tokens/call 58  s/call 2.45

G06 Q: What is form LP-14 for?
    A: Form LP-14 is a lightweight and durable backpack designed specifically for hikers and outdoor enthusiasts who need to carry their gear efficiently while maintaining comfo
G01 Q: What does error code E-118 mean?
    A: Error code E-118 typically indicates a problem with your computer's network connection or settings. It could be due to issues like incorrect DNS settings, firewall blocki
G32 Q: What is GW-OVR?
    A: GW-OVR stands for "Gear with Overdrive," a unique feature offered by Northwind Outfitters that allows customers to customize their gear purchases with overdrive effects o
G35 Q: Can I rent a tent from Northwind for a weekend instead of buying one?
    A: Yes, you can certainly rent a tent from Northwind Outfitters for your weekend trip. They offer various types and sizes of tents to suit different needs and preferences. J
G

### 1.2 — Open book by hand: paste the right article

> **Lecture:** Block 1 · *Put the right page in the prompt*

Now the opposite extreme, done by hand: for each answerable question, paste the article(s) that answer
it into the user message, and add three rules to the system message — answer only from the sources,
cite the source id, and say so when the sources do not contain the answer. This is **prompt
augmentation**: the facts arrive in the window at call time; the model's weights do not change.

It is not a fair system — somebody had to know which article answers each question — but it is the
**ceiling**: what this small model can do when the right page is in front of it. Everything later is
about getting close to it automatically.

In [7]:
SYS_RAG = ("You are the customer support assistant for Northwind Outfitters, an online outdoor-gear retailer. "
           "Answer the customer's question using ONLY the sources below. Cite the source id in square brackets, "
           "like [KB-004]. If the sources do not contain the answer, say you do not have that information and "
           "suggest contacting support. Answer in two or three sentences.")


def augmented(question: str, passages: list[tuple[str, str]]) -> str:
    """The user message: the sources, each labelled with its id, then the question."""
    sources = "\n\n".join(f"[{doc_id}]\n{text}" for doc_id, text in passages)
    return f"SOURCES:\n{sources}\n\nQUESTION: {question}"


oracle = run("the right article, pasted by hand",
             lambda q: (SYS_RAG, augmented(q.question, [(r, BY_ID[r]["text"]) for r in q.relevant]), q.relevant),
             questions=ANSWERABLE)
S_ORACLE = show(oracle)
print(f"\nclosed book: {S_CLOSED['correct']} of {S_CLOSED['of']}  ->  right article pasted: {S_ORACLE['correct']} of {S_ORACLE['of']}")
for qid in ["G06", "G01", "G10"]:
    print(f"{qid} A: {oracle.set_index('qid').loc[qid, 'answer'][:170]}")

the right article, pasted by hand  correct 22/27  stale 0/3  tokens/call 518  s/call 2.70

closed book: 2 of 27  ->  right article pasted: 22 of 27
G06 A: Form LP-14 is used to file a claim when a package is marked as delivered but not received. It requires details such as the NWT-tracking number, delivery address, and a de
G01 A: Error code E-118 indicates that the reset link has expired. This means that the time limit for using the reset link has been reached, typically after 30 minutes. To resol
G10 A: According to the return policy, you have 60 days from the delivery date to return most items for a full refund. This period runs from delivery, not from the purchase date


### 1.3 — Why not paste the whole help centre?

> **Lecture:** Block 1 · *Why not paste the whole help centre?*

If pasting the right article works, why not paste **all 34** into every call and let the model find
the answer? The window is big enough: 32,768 tokens for this model. Three reasons not to, and the
experiment below measures all three:

1. **Cost and time.** Every call now carries the whole help centre. The model reads (and a rented
   model bills) every one of those tokens on every question.
2. **Everything includes the things it must not see.** The three internal pages and the superseded
   policy go in too.
3. **It does not scale.** 34 articles fit. A real help centre has thousands; at this one's size per
   article (the cell prints it), ten thousand articles would be about three million tokens on every
   call.

In [8]:
ALL_SOURCES = [(d["doc_id"], d["text"]) for d in DOCS]
ALL_IDS = [d["doc_id"] for d in DOCS]
# About 10,500 tokens per call. On a GPU each answer takes 10-30 seconds; on a CPU, minutes. So on a CPU only
# the eight stale and restricted questions are asked (enough to see the leak and the old policy).
PASTE_QUESTIONS = GOLD if device.type != "cpu" else GOLD[GOLD.qtype.isin(["stale", "restricted"])]
if device.type == "cpu":
    print("CPU only: asking the 8 stale + restricted questions, not all 35 (see the comment above)\n")


def paste_everything(q):
    return SYS_RAG, augmented(q.question, ALL_SOURCES), ALL_IDS


try:
    everything = run("the whole help centre pasted", paste_everything, questions=PASTE_QUESTIONS)
except (torch.OutOfMemoryError, RuntimeError) as err:      # CUDA raises the first; a Mac's GPU, the second
    if "out of memory" not in str(err).lower():
        raise
    print("OUT OF MEMORY: 10,500 tokens per call did not fit on this machine's GPU. That is the third reason.")
    print("Set FORCE_CPU = True in the first cell and run the notebook again: on the CPU only 8 questions are asked here.")
    raise
S_EVERY = show(everything)
ALL_SRC_TOK = n_tokens("\n\n".join(f"[{d}]\n{t}" for d, t in ALL_SOURCES))     # the sources part of the prompt
print(f"\nthe whole help centre: {ALL_SRC_TOK:,} tokens of sources on every call "
      f"(about {ALL_SRC_TOK // len(DOCS)} per article)")
print("internal facts given out:", ", ".join(everything[everything.leaked].qid) or "none")
print("old-policy answers      :", ", ".join(everything[everything.stale.eq(True)].qid) or "none")

OUT OF MEMORY: 10,500 tokens per call did not fit on this machine's GPU. That is the third reason.
Set FORCE_CPU = True in the first cell and run the notebook again: on the CPU only 8 questions are asked here.


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.95 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.61 GiB is free. Including non-PyTorch memory, this process has 11.96 GiB memory in use. Of the allocated memory 11.69 GiB is allocated by PyTorch, and 144.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

---

# 2 — Retrieval: find the right page automatically

> **Lecture:** Block 2 · *RAG: search first, then answer*

**Retrieval-augmented generation (RAG)** automates §1.2. It has two halves:

- **Indexing — once, ahead of time.** Cut every article into passages, turn each passage into a
  vector, and store the vectors with the passage's text and its labels (which article, audience,
  status).
- **Answering — on every question.** Turn the question into a vector the same way, find the passages
  nearest to it, put the best few into the prompt (§1.2's augmented prompt), and let the model write.

The model is untouched; everything new happens *before* the model call. This section builds the
search half and measures it **on its own**, before any answer is written — the first of the two numbers
a RAG system needs.

### 2.1 — Chunking: splitting pages into passages

> **Lecture:** Block 2 · *Chunking: four ways to split a page* · Appendix · *This lecture's split: one passage per section*

Why not one vector per article? Three reasons. An encoder reads only so much text: this one stops at
256 tokens and ignores the rest. One vector for a long article is a summary, and a summary can blur the
one paragraph you need. And the window should carry the paragraph, not the article. So the articles are
cut into **passages** ("chunks"). There are four common ways to choose where to cut:

- **Fixed size** — every *n* characters (or tokens), wherever that falls. It needs to know nothing
  about the text, and it cuts through the middle of sentences.
- **Fixed size with overlap** — the same, but each passage starts a little before the previous one
  ended, so a phrase shorter than the overlap that one boundary cuts is whole in the passage beside it.
  The repeated text is stored and embedded twice, and can reach the window twice.
- **By structure** — cut where the writers did: at headings, paragraphs or list items. It needs
  documents that have a structure.
- **By meaning ("semantic chunking")** — embed every sentence, and start a new passage where the next
  sentence's meaning moves away from the sentence before it. It needs no headings (the cutter below
  is not shown them); it costs one embedding per sentence before a single passage exists, and it has a
  threshold to tune.

This notebook indexes the help centre **by structure**: one passage per `##` section, each carrying
its article's title so it still makes sense on its own. Most articles have nothing before their first
section except the title line; a passage that is only a title answers nothing and would take a place
in the window, so it is skipped (every section already carries the title). The other three cutters are
defined in the same cell, and §2.6 scores all four on the golden questions, once §2.3 has given us a
way to score a search.

In [ ]:
def chunk_by_section(doc: dict) -> list[dict]:
    """One passage per '## ' section; each carries the article title so it stands on its own.
    The part before the first section is kept only if it says more than the title."""
    parts = re.split(r"\n(?=## )", doc["text"])
    passages = []
    for i, part in enumerate(parts):
        part = part.strip()
        if i == 0 and "\n" not in part:
            continue                                   # the title line alone: nothing to find
        body = part if i == 0 else f"# {doc['title']}\n{part}"
        passages.append(dict(doc_id=doc["doc_id"], section=i, text=body,
                             audience=doc["audience"], status=doc["status"]))
    return passages


def chunk_fixed(doc: dict, size: int, overlap: int = 0) -> list[dict]:
    """Every `size` characters, wherever that falls: the simplest cut, and the bluntest.
    With `overlap`, each passage starts `overlap` characters before the previous one ended."""
    assert 0 <= overlap < size
    t, step = doc["text"], size - overlap
    return [dict(doc_id=doc["doc_id"], section=i, text=t[s:s + size], audience=doc["audience"], status=doc["status"])
            for i, s in enumerate(range(0, max(len(t) - overlap, 1), step))]


def split_sentences(text: str) -> list[str]:
    """Sentences, headings and list items, in order. Joined back together they are the original text."""
    ends = [m.end() for m in re.finditer(r"(?<=[.!?])(?<!\n\d\.)\s+|\n\s*\n|\n(?=- )", text)] + [len(text)]
    units, start = [], 0
    for end in sorted(set(ends)):
        if text[start:end].strip():
            units.append(text[start:end])
            start = end
    return units


def is_heading(unit: str) -> bool:
    return unit.lstrip().startswith("#")


# The semantic cut needs every sentence as a vector first: one pass of the encoder over the whole help centre.
# Headings are left out of the comparison on purpose: this cutter has to find the changes of subject by itself.
SENTENCES = {d["doc_id"]: split_sentences(d["text"]) for d in DOCS}
SENT_VECS = {k: EMBEDDER.encode([u for u in v if not is_heading(u)], normalize_embeddings=True, convert_to_numpy=True)
             for k, v in SENTENCES.items()}
ADJACENT = np.concatenate([np.sum(v[1:] * v[:-1], axis=1) for v in SENT_VECS.values()])   # each sentence against the next
SEMANTIC_CUT = float(np.percentile(ADJACENT, 33))     # cut where the similarity is in the lowest third


def chunk_semantic(doc: dict, threshold: float = SEMANTIC_CUT, min_chars: int = 200) -> list[dict]:
    """Start a new passage where the meaning moves: where a sentence is less similar than `threshold`
    to the sentence before it. Headings are not compared (a heading travels with the sentence that
    follows it), and a passage shorter than `min_chars` is never closed."""
    vecs = iter(SENT_VECS[doc["doc_id"]])
    groups, current, waiting, last = [], [], [], None
    for unit in SENTENCES[doc["doc_id"]]:
        if is_heading(unit):
            waiting.append(unit)                       # placed by the sentence after it, not by what it says
            continue
        vec = next(vecs)
        if last is not None and float(vec @ last) < threshold and sum(map(len, current)) >= min_chars:
            groups.append(current)
            current = []
        current += waiting + [unit]
        waiting, last = [], vec
    groups.append(current + waiting)
    return [dict(doc_id=doc["doc_id"], section=i, text="".join(g).strip(), audience=doc["audience"], status=doc["status"])
            for i, g in enumerate(groups)]


PASSAGES = [p for d in DOCS for p in chunk_by_section(d)]
lens = [n_tokens(p["text"]) for p in PASSAGES]
n_title_only = sum("\n" not in re.split(r"\n(?=## )", d["text"])[0].strip() for d in DOCS)
print(f"{len(DOCS)} articles -> {len(PASSAGES)} passages, "
      f"{min(lens)}-{max(lens)} tokens each (median {int(np.median(lens))})")
print(f"skipped: {n_title_only} parts that were only an article's title")
print("\nKB-001, cut by section:")
for p in PASSAGES:
    if p["doc_id"] == "KB-001":
        head = p["text"].splitlines()[1 if p["section"] else 0]
        print(f"  passage {p['section']}: {n_tokens(p['text']):3} tokens  {head}")

# The same article under the other three cutters (the slide draws these boundaries).
one = BY_ID["KB-001"]
ways = {"fixed size, 500 characters": chunk_fixed(one, 500),
        "fixed size, 500 with 100 overlap": chunk_fixed(one, 500, 100),
        "by meaning (semantic)": chunk_semantic(one),
        "by structure, one per section": [p for p in PASSAGES if p["doc_id"] == "KB-001"]}
print(f"\nKB-001 ({len(one['text']):,} characters, {len(SENT_VECS['KB-001'])} sentences), cut four ways:")
for name, ps in ways.items():
    print(f"  {name:34} {len(ps)} passages")
first, second = ways["fixed size, 500 characters"][:2]
print("\nthe first fixed-size boundary falls here:  ..." + first["text"][-42:].replace("\n", " ") + " | "
      + second["text"][:42].replace("\n", " ") + "...")
where = lambda ps: [[one["text"].index(p["text"]), one["text"].index(p["text"]) + len(p["text"])] for p in ps]
print("CUTS=" + json.dumps({"n": len(one["text"]), "fixed": where(ways["fixed size, 500 characters"]),
                            "overlap": where(ways["fixed size, 500 with 100 overlap"]),
                            "semantic": where(ways["by meaning (semantic)"]),
                            "section": [m.start() + 1 for m in re.finditer(r"\n(?=## )", one["text"])]}))
print(f"semantic cut: {len(ADJACENT)} sentence-to-sentence similarities in the help centre; "
      f"a new passage starts below {SEMANTIC_CUT:.2f} (the lowest third)")
n_cuts = n_at_heading = n_headings = 0
for d in DOCS:
    heads = [m.start() + 1 for m in re.finditer(r"\n(?=## )", d["text"])]
    starts = [d["text"].index(p["text"]) for p in chunk_semantic(d)][1:]
    n_cuts, n_headings = n_cuts + len(starts), n_headings + len(heads)
    n_at_heading += sum(start in heads for start in starts)
print(f"across the help centre the semantic cutter, which is never shown a heading, made {n_cuts} cuts; "
      f"{n_at_heading} of them fell at one of the {n_headings} section headings")

34 articles -> 142 passages, 22-177 tokens each (median 70)
skipped: 25 parts that were only an article's title

KB-001, cut by section:
  passage 0:  49 tokens  # Standard Shipping Options and Delivery Windows
  passage 1: 147 tokens  ## Services and delivery windows
  passage 2: 116 tokens  ## Fees
  passage 3: 143 tokens  ## Surcharges
  passage 4:  77 tokens  ## Addresses we cannot ship to

KB-001 (2,147 characters, 21 sentences), cut four ways:
  fixed size, 500 characters         5 passages
  fixed size, 500 with 100 overlap   6 passages
  by meaning (semantic)              3 passages
  by structure, one per section      5 passages

the first fixed-size boundary falls here:  ...ess day. Delivery windows below are counte | d in business days from the day the parcel...
CUTS={"n": 2147, "fixed": [[0, 500], [500, 1000], [1000, 1500], [1500, 2000], [2000, 2147]], "overlap": [[0, 500], [400, 900], [800, 1300], [1200, 1700], [1600, 2100], [2000, 2147]], "semantic": [[0, 312], [313, 1840

### 2.2 — Embeddings turn text into numbers

> **Lecture:** Block 2 · *Embeddings turn text into numbers* · Block 2 · *Search by meaning: find the closest passages*

Week 5's encoder turns each passage into **384 numbers**, arranged by training so that texts with the
same meaning land close together. The question goes through the same encoder. Search is then Week 2's
cosine similarity: because the vectors are scaled to length 1, the dot product *is* the cosine, and
scoring the question against every passage at once is **one matrix multiplication**.

That is all a "vector search" is at this size. Vector databases (Chroma, pgvector, Pinecone and others)
and index libraries such as FAISS exist for millions of passages, where they trade a little accuracy for a lot of speed with
*approximate* search, and they do more than search (§2.7).

In [ ]:
E = EMBEDDER.encode([p["text"] for p in PASSAGES], normalize_embeddings=True,
                    batch_size=64, convert_to_numpy=True)


def embed(text: str) -> np.ndarray:
    return EMBEDDER.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0]


print("passage vectors:", E.shape, "  length of one:", round(float(np.linalg.norm(E[0])), 3))
q = GOLD.set_index("qid").loc["G09", "question"]
qv = embed(q)                         # the question through the same encoder
t0 = time.perf_counter()
scores = E @ qv                       # cosine with every passage, in one multiplication
ms = (time.perf_counter() - t0) * 1000
print(f"\nQ: {q}")
print(f"scored all {len(PASSAGES)} passages in {ms:.3f} ms (the multiplication alone); the five nearest:")
for i in np.argsort(-scores)[:5]:
    print(f"  {scores[i]:.3f}  {PASSAGES[i]['doc_id']}  {PASSAGES[i]['text'].splitlines()[1 if PASSAGES[i]['section'] else 0]}")

passage vectors: (142, 384)   length of one: 1.0

Q: The courier says it turned up but there is nothing on my doorstep. Now what?
scored all 142 passages in 0.044 ms (the multiplication alone); the five nearest:
  0.525  KB-003  ## Marked delivered but not received
  0.466  KB-003  # Lost, Stolen, and Undelivered Packages
  0.383  KB-019  ## If you think someone else has been in your account
  0.364  KB-002  ## Reading a tracking status
  0.334  KB-002  ## When tracking has not moved


**A picture of the space.** 384 numbers cannot be drawn, so the cell below flattens them to two with
principal component analysis (PCA, a cousin of Week 2's SVD): the two directions along which the
passages differ most. Distances are distorted — two dimensions keep only a small share of the
spread. The broad grouping by topic survives (products and account pages most clearly); exact
distances do not, so the question's three nearest passages in the full space, circled on the slide,
need not look nearest on the flat picture. The coordinates are printed so the slide can draw it.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=SEED).fit(E)
xy = pca.transform(E)
qxy = pca.transform(embed(q)[None, :])[0]
top3 = [int(i) for i in np.argsort(-scores)[:3]]
print(f"share of the spread kept by 2 dimensions: {pca.explained_variance_ratio_.sum():.0%}")
pts = [[BY_ID[p["doc_id"]]["topic"], round(float(x), 3), round(float(y), 3)] for p, (x, y) in zip(PASSAGES, xy)]
print("MAP_JSON=" + json.dumps({"points": pts, "question": [round(float(qxy[0]), 3), round(float(qxy[1]), 3)],
                                 "top3": top3, "qid": "G09"}, separators=(",", ":")))

share of the spread kept by 2 dimensions: 14%
MAP_JSON={"points":[["shipping",-0.011,0.332],["shipping",-0.253,0.126],["shipping",-0.126,0.385],["shipping",-0.074,0.35],["shipping",-0.205,0.196],["shipping",-0.092,0.127],["shipping",-0.18,-0.167],["shipping",-0.14,-0.094],["shipping",-0.158,0.021],["shipping",-0.269,-0.078],["shipping",-0.069,-0.017],["shipping",-0.196,-0.03],["returns",0.206,0.134],["returns",-0.297,0.254],["returns",0.145,0.206],["returns",0.038,0.267],["returns",-0.242,-0.047],["returns",-0.087,0.041],["returns",-0.342,0.127],["returns",-0.282,-0.077],["returns",-0.185,-0.116],["returns",-0.176,0.419],["returns",-0.246,0.394],["returns",-0.355,0.037],["returns",-0.183,0.285],["returns",-0.214,0.249],["returns",-0.188,0.201],["returns",-0.193,0.242],["returns",-0.13,0.414],["returns",-0.319,0.109],["returns",-0.361,0.233],["returns",-0.152,0.409],["returns",-0.163,0.124],["warranty",0.326,0.115],["warranty",0.417,0.102],["warranty",0.637,0.098],["warranty",0.602,0.21

### 2.3 — Measure the search separately from the answers

> **Lecture:** Block 2 · *Measure the search separately from the answers*

Before the model writes a word, ask two questions of the search, on every answerable golden question
(Week 2's retrieval measures, with Week 3's discipline: a golden set, decided in advance):

- **recall@k** — of the article(s) that answer the question, what share came back in the top *k*
  passages? ("Did the right page come back?")
- **fact in the window** — is the exact phrase the answer needs (`answer_span`) inside one of the *k*
  passages *from an article that answers the question*? ("Could the model possibly get it right?")
  The phrase is only counted in an answering article, because a common phrase such as "60 days" also
  appears in unrelated articles.

The two disagree in both directions. The right article can come back as the wrong section (recall
counts it; the fact is missing). And a question that needs two articles can have its phrase in the one
that came back (recall is a half; the fact is there).

A RAG system has **two** numbers — one for finding, one for answering — and they fail for different
reasons. Measuring them separately is how you know which half to fix.

In [ ]:
def recall_at_k(found_ids: list[str], relevant: list[str]) -> float:
    return len(set(found_ids) & set(relevant)) / len(relevant)


def fact_in_window(idx: list[int], q, passages=None) -> bool:
    """Is the answer phrase inside a retrieved passage from an article that answers the question?"""
    passages = passages or PASSAGES
    return any(passages[i]["doc_id"] in q.relevant and q.answer_span in passages[i]["text"] for i in idx)


def score_search(search, k: int = 5, questions=ANSWERABLE) -> pd.DataFrame:
    rows = []
    for _, q in questions.iterrows():
        idx = search(q.question, k)
        found = [PASSAGES[i]["doc_id"] for i in idx]
        rows.append(dict(qid=q.qid, qtype=q.qtype, recall=recall_at_k(found, q.relevant),
                         fact=fact_in_window(idx, q)))
    return pd.DataFrame(rows)


def allowed_ids(allowed=None):
    return [i for i, p in enumerate(PASSAGES) if allowed is None or allowed(p)]


def search_meaning(question: str, k: int, allowed=None) -> list[int]:
    """Dense search: the k passages whose vectors are nearest the question's."""
    ids = allowed_ids(allowed)
    s = E[ids] @ embed(question)
    return [ids[i] for i in np.argsort(-s)[:k]]


meaning = score_search(search_meaning, k=5)
print(f"search by meaning, k = 5:  recall@5 {meaning.recall.mean():.2f}   fact in the window {meaning.fact.mean():.2f}")
print("\nwhere it misses (recall@5 by kind of question):")
print(meaning.groupby("qtype").recall.mean().round(2).to_string())

search by meaning, k = 5:  recall@5 0.89   fact in the window 0.85

where it misses (recall@5 by kind of question):
qtype
cross_section    1.00
lexical          0.88
multi_doc        0.60
semantic         1.00
stale            1.00


### 2.4 — Two kinds of search: by keyword and by meaning

> **Lecture:** Block 2 · *Two kinds of search: by keyword and by meaning*

Meaning search has a blind spot: **exact codes and terms**. To an encoder trained on general English,
`E-118`, `WC-22` and `TH45-B` are rare scraps of characters, and a house term such as *Bulky Item* is
two ordinary words; their vectors land in a vague neighbourhood. A customer who types the code or the
term wants the page with *that string*. On this small help centre the gap is one question in eight
(the cell prints which); it widens as a collection fills with look-alike codes.

**Keyword search** finds exactly that. BM25, the standard keyword ranking since the 1990s, is Week 2's
TF-IDF grown up: a passage scores for every word it shares with the question, more for rare words,
with a cap so one word repeated twenty times does not win. Its blind spot is the mirror image:
**paraphrase**. "The courier says it turned up but there is nothing on my doorstep" shares almost no
words with an article titled *Lost, Stolen, and Undelivered Packages*.

In [ ]:
from rank_bm25 import BM25Okapi


def words(text: str) -> list[str]:
    """Lower-case words, keeping codes like e-118 and th45-b in one piece."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


BM25 = BM25Okapi([words(p["text"]) for p in PASSAGES])


def search_keywords(question: str, k: int, allowed=None) -> list[int]:
    ids = allowed_ids(allowed)
    s = BM25.get_scores(words(question))[ids]
    return [ids[i] for i in np.argsort(-s)[:k]]


keyword = score_search(search_keywords, k=5)
both = pd.DataFrame({"keyword (BM25)": keyword.groupby("qtype").recall.mean(),
                     "meaning (embeddings)": meaning.groupby("qtype").recall.mean()}).round(2)
print("recall@5 by kind of question")
print(both.to_string())
missed = [q for _, q in ANSWERABLE[ANSWERABLE.qtype == "lexical"].iterrows()
          if not {PASSAGES[i]["doc_id"] for i in search_meaning(q.question, 5)} & set(q.relevant)]
print("\nexact-term questions whose article meaning search misses at k = 5:",
      "; ".join(f"{q.qid} {q.question}" for q in missed) or "none")
for qid in ["G01", "G09"]:
    qq = GOLD.set_index("qid").loc[qid, "question"]
    kw = [PASSAGES[i]["doc_id"] for i in search_keywords(qq, 5)]
    mn = [PASSAGES[i]["doc_id"] for i in search_meaning(qq, 5)]
    print(f"\n{qid} {qq}\n   answer in {GOLD.set_index('qid').loc[qid, 'relevant']}"
          f"\n   keyword top 5: {kw}\n   meaning top 5: {mn}")

recall@5 by kind of question
               keyword (BM25)  meaning (embeddings)
qtype                                              
cross_section            1.00                  1.00
lexical                  1.00                  0.88
multi_doc                0.60                  0.60
semantic                 0.12                  1.00
stale                    0.67                  1.00

exact-term questions whose article meaning search misses at k = 5: G02 What is the Bulky Item badge on a product page?

G01 What does error code E-118 mean?
   answer in ['KB-020']
   keyword top 5: ['KB-020', 'KB-019', 'KB-042', 'KB-015', 'KB-014']
   meaning top 5: ['KB-019', 'KB-031', 'KB-020', 'KB-031', 'KB-042']

G09 The courier says it turned up but there is nothing on my doorstep. Now what?
   answer in ['KB-003']
   keyword top 5: ['KB-019', 'KB-026', 'KB-018', 'KB-002', 'KB-027']
   meaning top 5: ['KB-003', 'KB-003', 'KB-019', 'KB-002', 'KB-002']


### 2.5 — Hybrid search: use both, then re-rank

> **Lecture:** Block 2 · *Hybrid search: use both, then re-rank*

Two searches with opposite blind spots: run both and **merge the lists**. Reciprocal rank fusion (RRF)
merges by rank, not by score (the two scores are on unrelated scales): a passage gets
1 / (60 + its rank) from each list, and the totals are sorted. A passage near the top of either list
rises; a passage near the top of both rises furthest.

Then **re-rank**. Both searches compared the question with each passage *separately* (one vector each,
or word counts). A **cross-encoder** reads the question and one passage *together*, as one input, and
outputs one relevance score — much more accurate, and far too slow to run on a whole collection. So it
reads only the merged shortlist of 20 and picks the best 5. Fast-and-rough to find candidates,
slow-and-careful to choose among them: the standard shape of a production search.

In [ ]:
def rrf(*rankings: list[int], c: int = 60) -> list[int]:
    """Reciprocal rank fusion: each list gives 1 / (c + rank); sort by the total."""
    total = defaultdict(float)
    for ranking in rankings:
        for rank, i in enumerate(ranking, start=1):
            total[i] += 1 / (c + rank)
    return [i for i, _ in sorted(total.items(), key=lambda kv: -kv[1])]


def search_hybrid(question: str, k: int, allowed=None) -> list[int]:
    return rrf(search_keywords(question, 20, allowed), search_meaning(question, 20, allowed))[:k]


def search_rerank(question: str, k: int, allowed=None) -> list[int]:
    """Hybrid shortlist of 20, re-read by the cross-encoder, best k kept."""
    shortlist = search_hybrid(question, 20, allowed)
    s = RERANKER.predict([(question, PASSAGES[i]["text"]) for i in shortlist])
    return [shortlist[i] for i in np.argsort(-s)[:k]]


SEARCHES = {"keyword (BM25)": search_keywords, "meaning (embeddings)": search_meaning,
            "both, fused (RRF)": search_hybrid, "fused, then re-ranked": search_rerank}
table = []
for name, fn in SEARCHES.items():
    r = score_search(fn, k=5)
    table.append(dict(search=name, **{"recall@5": r.recall.mean(), "fact in window": r.fact.mean(),
                                       "codes": r[r.qtype == "lexical"].recall.mean(),
                                       "paraphrases": r[r.qtype == "semantic"].recall.mean()}))
SEARCH_TABLE = pd.DataFrame(table).set_index("search").round(2)
print("top 5 passages, the 27 answerable golden questions")
print(SEARCH_TABLE.to_string())

top 5 passages, the 27 answerable golden questions
                       recall@5  fact in window  codes  paraphrases
search                                                             
keyword (BM25)             0.63            0.59   1.00         0.12
meaning (embeddings)       0.89            0.85   0.88         1.00
both, fused (RRF)          0.87            0.89   1.00         0.75
fused, then re-ranked      0.91            0.93   1.00         0.88


**How many passages to keep?** The deeper you look, the more often the right page is in the window —
and the more the model has to read. The curve below is for the search chosen above.

In [ ]:
curve = []
for k in (1, 3, 5, 10, 20):
    r = score_search(search_rerank, k=k)
    curve.append(dict(k=k, recall=round(r.recall.mean(), 2), fact=round(r.fact.mean(), 2)))
RECALL_CURVE = pd.DataFrame(curve).set_index("k")
print("fused + re-ranked search: recall@k and fact-in-window, by k")
print(RECALL_CURVE.to_string())

fused + re-ranked search: recall@k and fact-in-window, by k
    recall  fact
k               
1     0.63  0.56
3     0.89  0.89
5     0.91  0.93
10    0.94  0.93
20    1.00  1.00


### 2.6 — Which way of chunking works best?

> **Lecture:** Block 2 · *Which way of chunking works best?*

Back to the first decision, now that a search can be scored. The same meaning search (§2.3), the same
27 questions, and the help centre cut seven ways with §2.1's four cutters. Four columns to read:

- **answers whole** — of the 27 answer phrases, how many survive the cut in one piece in some passage.
  A fixed-size boundary can fall inside the phrase; overlap exists to repair exactly that.
- **recall@5** — the right *article* came back.
- **fact in window** — the right *part* of it came back. The right article is not enough if the
  passage that came back is the wrong half of it.
- **tokens in window** — what five passages cost on every call.

Then one more line for the semantic cut: its threshold is a setting, and the result moves with it.

In [ ]:
def answers_whole(passages: list[dict]) -> int:
    """Of the answerable questions, how many answer phrases are still in one piece in some passage."""
    return sum(any(p["doc_id"] in q.relevant and q.answer_span in p["text"] for p in passages)
               for _, q in ANSWERABLE.iterrows())


def score_cutting(passages: list[dict], k: int = 5) -> dict:
    vecs = EMBEDDER.encode([p["text"] for p in passages], normalize_embeddings=True, batch_size=64, convert_to_numpy=True)
    rec, fact, toks = [], [], []
    for _, q in ANSWERABLE.iterrows():
        idx = np.argsort(-(vecs @ embed(q.question)))[:k]
        window = "\n".join(passages[i]["text"] for i in idx)
        rec.append(recall_at_k([passages[i]["doc_id"] for i in idx], q.relevant))
        fact.append(fact_in_window(idx, q, passages))
        toks.append(n_tokens(window))
    return {"passages": len(passages), "answers whole": answers_whole(passages), "recall@5": round(np.mean(rec), 2),
            "fact in window": round(np.mean(fact), 2), "tokens in window": int(np.mean(toks))}


def every(cut) -> list[dict]:
    return [p for d in DOCS for p in cut(d)]


cuts = {"200 characters": every(lambda d: chunk_fixed(d, 200)),
        "200 characters, 50 overlap": every(lambda d: chunk_fixed(d, 200, 50)),
        "500 characters": every(lambda d: chunk_fixed(d, 500)),
        "500 characters, 100 overlap": every(lambda d: chunk_fixed(d, 500, 100)),
        "by meaning (semantic)": every(chunk_semantic),
        "one per section": PASSAGES,
        "one per article": [dict(doc_id=d["doc_id"], section=0, text=d["text"], audience=d["audience"], status=d["status"]) for d in DOCS]}
CHUNK_TABLE = pd.DataFrame({name: score_cutting(ps) for name, ps in cuts.items()}).T
print(f"search by meaning, k = 5, the same {len(ANSWERABLE)} questions, cut seven ways")
print(CHUNK_TABLE.to_string())
print(f"\nthe semantic cut embedded {sum(map(len, SENT_VECS.values()))} sentences to choose its boundaries; "
      "the section cut embedded none. Its threshold moves the result:")
for pct in (20, 33, 45):
    threshold = float(np.percentile(ADJACENT, pct))
    r = score_cutting(every(lambda d: chunk_semantic(d, threshold)))
    print(f"  SEMANTIC cut at the lowest {pct}% of similarities: {r['passages']:3} passages   "
          f"recall@5 {r['recall@5']:.2f}   fact in window {r['fact in window']:.2f}")
limit = EMBEDDER.max_seq_length                                # how much of a passage the encoder reads
too_long = lambda ps: sum(len(EMBEDDER.tokenizer(p["text"], verbose=False).input_ids) > limit for p in ps)
print(f"\nthe encoder reads the first {limit} tokens of a passage and ignores the rest. Longer than that: "
      f"{too_long(cuts['one per article'])} of {len(DOCS)} whole articles, {too_long(PASSAGES)} of {len(PASSAGES)} sections, "
      f"{too_long(cuts['by meaning (semantic)'])} of {len(cuts['by meaning (semantic)'])} semantic passages")

search by meaning, k = 5, the same 27 questions, cut seven ways
                             passages  answers whole  recall@5  fact in window  tokens in window
200 characters                  244.0           25.0      0.81            0.59             221.0
200 characters, 50 overlap      307.0           27.0      0.81            0.67             226.0
500 characters                  111.0           26.0      0.91            0.81             489.0
500 characters, 100 overlap     120.0           27.0      0.91            0.89             525.0
by meaning (semantic)           113.0           27.0      0.91            0.81             576.0
one per section                 142.0           27.0      0.89            0.85             432.0
one per article                  34.0           27.0      0.91            0.93            1543.0

the semantic cut embedded 458 sentences to choose its boundaries; the section cut embedded none. Its threshold moves the result:


  SEMANTIC cut at the lowest 20% of similarities:  90 passages   recall@5 0.83   fact in window 0.74


  SEMANTIC cut at the lowest 33% of similarities: 113 passages   recall@5 0.91   fact in window 0.81


  SEMANTIC cut at the lowest 45% of similarities: 130 passages   recall@5 0.83   fact in window 0.81

the encoder reads the first 256 tokens of a passage and ignores the rest. Longer than that: 23 of 34 whole articles, 0 of 142 sections, 1 of 113 semantic passages


Read the table before you take a rule from it. Overlap leaves every answer phrase whole and lifts
**fact in window** at both sizes, for more passages to store (120 against 111 at 500 characters, 307
against 244 at 200). It does not move recall: the right article was already coming back, and overlap
changes which part of it comes back. The semantic cut found its boundaries from the sentences alone,
with no headings to help, after embedding every sentence; read its row against the section cut's, and
then read the three lines under the table, because its threshold moves the result. Where a collection
has no headings to use, that row is the one that matters, and a fixed size with overlap is the sensible
default to compare it with.

This help centre has headings, so the rest of the notebook keeps **one passage per section**: within a
few points of the best fixed cut, for fewer tokens in the window, and each passage is something a
person can read, cite and check. (It is also the only cut here that carries the article's title on
every passage, which is part of what it is being credited with.) Whole articles keep the most facts in
the window here, at more than three times the tokens, and with a catch the table cannot show: the
encoder read only the top of most of them (the last line of the output). These articles are short
enough to get away with it.

### 2.7 — A vector database stores the embeddings

> **Lecture:** Block 2 · *A vector database stores the embeddings*

So far the "index" is a NumPy array in memory: 142 rows of 384 numbers, gone when the
kernel stops. A **vector database** replaces it in a system that has to keep running. It does four jobs:

1. **Keeps** each passage's vector together with its text and its labels (article, audience, status)
   on disk, so nothing is embedded twice.
2. **Searches** for the nearest vectors with an index built for it (usually a graph of neighbours,
   HNSW) that checks only the most promising neighbourhoods instead of every vector. That is
   *approximate* by design, and it is what makes millions of vectors searchable.
3. **Filters by label inside the search** (`audience = public AND status = current`): the feature §4
   depends on.
4. **Changes** one passage at a time: add, update or delete an article without rebuilding everything.

The cell below uses **Chroma**, an open-source vector database that runs inside the Python process and
keeps its data in a folder. Others you will meet: **FAISS** (a search-index library rather than a
database: it holds vectors and ids, and you keep the text and labels yourself), **pgvector** (a vector
column with nearest-neighbour search inside PostgreSQL, beside your ordinary tables), servers you can
run or rent, such as Weaviate, Qdrant and Milvus, and Pinecone, which is rented only. They differ in
where they run and how far they scale; FAISS apart, all of them do the four jobs above.

The rest of this notebook keeps the NumPy array, so that every later number depends on one
multiplication and nothing else. The cell checks that, at this size, Chroma's index returns the same
passages as checking every vector.

In [ ]:
try:
    import chromadb
    import tempfile

    CHROMA_DIR = Path(tempfile.gettempdir()) / "northwind_chroma"
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))            # a folder on disk
    try:
        client.delete_collection("help_centre")                         # a re-run starts clean
    except Exception:
        pass                                                            # first run: nothing to delete
    col = client.create_collection("help_centre", metadata={"hnsw:space": "cosine"})
    col.add(ids=[f"p{i}" for i in range(len(PASSAGES))],
            embeddings=E.tolist(),
            documents=[p["text"] for p in PASSAGES],
            metadatas=[{"doc_id": p["doc_id"], "audience": p["audience"], "status": p["status"]} for p in PASSAGES])
    print(f"stored: {col.count()} passages, each with its vector, its text and three labels, in {CHROMA_DIR.name}/")

    # 1. The same search as numpy? Every answerable question, top 5, both ways (ids only from both).
    same, t_np, t_db = 0, 0.0, 0.0
    for _, q in ANSWERABLE.iterrows():
        qv = embed(q.question)
        t0 = time.perf_counter(); by_numpy = [int(i) for i in np.argsort(-(E @ qv))[:5]]; t_np += time.perf_counter() - t0
        t0 = time.perf_counter(); hit = col.query(query_embeddings=[qv.tolist()], n_results=5, include=[]); t_db += time.perf_counter() - t0
        same += by_numpy == [int(i[1:]) for i in hit["ids"][0]]
    print(f"same five passages in the same order as numpy: {same} of {len(ANSWERABLE)} questions")
    print(f"time per search: numpy {1000 * t_np / len(ANSWERABLE):.3f} ms, Chroma {1000 * t_db / len(ANSWERABLE):.3f} ms")

    # 2. A filter inside the search: the same internal-code question, with and without it.
    gw = embed("What is GW-OVR?").tolist()
    hit = col.query(query_embeddings=[gw], n_results=3)
    print(f"{'no filter':42}:", [m["doc_id"] for m in hit["metadatas"][0]])
    hit = col.query(query_embeddings=[gw], n_results=3, where={"$and": [{"audience": "public"}, {"status": "current"}]})
    print(f"{'where audience=public AND status=current':42}:", [m["doc_id"] for m in hit["metadatas"][0]])

    # 3. It is still there: a NEW Python process opens the folder and counts the passages. Nothing is embedded again.
    count = subprocess.run([sys.executable, "-c",
                            "import sys, chromadb; print(chromadb.PersistentClient(path=sys.argv[1]).get_collection('help_centre').count())",
                            str(CHROMA_DIR)], capture_output=True, text=True).stdout.strip()
    print(f"a new Python process, opening the same folder, finds: {count} passages")
except ImportError:
    print("chromadb is not installed (pip install chromadb); nothing later in this notebook depends on this cell")

stored: 142 passages, each with its vector, its text and three labels, in northwind_chroma/
same five passages in the same order as numpy: 27 of 27 questions
time per search: numpy 0.014 ms, Chroma 0.289 ms
no filter                                 : ['KB-041', 'KB-041', 'KB-041']
where audience=public AND status=current  : ['KB-011', 'KB-010', 'KB-010']


a new Python process, opening the same folder, finds: 142 passages


---

# 3 — Answering from the page

> **Lecture:** Block 3 · *The augmented prompt: rules, sources, question*

### 3.1 — The augmented prompt

Retrieval hands the model a few passages; the prompt decides what it does with them. Three parts, in
this order:

1. **The rules** (system message): who it is, *answer only from the sources*, *cite the source id*,
   *say so when the sources do not contain the answer*. Stable across calls — the same on every
   question.
2. **The sources** (start of the user message): each passage labelled with its article id, so the
   answer can cite it and a person can check it.
3. **The question**, last, nearest to where the model starts writing.

The cell below builds one real prompt and shows where its tokens go.

In [ ]:
def rag_prompt(q, k: int = 5, search=search_rerank, allowed=None, system: str = SYS_RAG):
    """(system, user, the article ids in the window) for one golden question."""
    idx = search(q.question, k, allowed)
    passages = [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx]
    return system, augmented(q.question, passages), [PASSAGES[i]["doc_id"] for i in idx]


one = GOLD.set_index("qid").loc["G22"]
one = one.copy(); one["qid"] = "G22"
sys_msg, user_msg, ctx = rag_prompt(one)
sources_part = user_msg.split("\n\nQUESTION:")[0]
print(f"Q: {one.question}\n")
print(f"rules (system message)   {n_tokens(sys_msg):5} tokens")
print(f"sources: {len(ctx)} passages    {n_tokens(sources_part):5} tokens   from {ctx}")
print(f"the question             {n_tokens(one.question):5} tokens")
print("the five passages: " + " | ".join(
    f"{PASSAGES[i]['doc_id']} {PASSAGES[i]['text'].splitlines()[1 if PASSAGES[i]['section'] else 0].lstrip('# ')}"
    for i in search_rerank(one.question, 5)))
answer, n_in, sec = ask(sys_msg, user_msg)
print(f"\nthe model read {n_in} tokens in all (with the chat template's role markers) and wrote, in {sec:.1f} s:")
print(answer)
print("\ngrade:", grade("G22", answer))

Q: What will an Expedited order to Alaska cost me in shipping?

rules (system message)      76 tokens
sources: 5 passages      509 tokens   from ['KB-001', 'KB-001', 'KB-006', 'KB-001', 'KB-013']
the question                13 tokens
the five passages: KB-001 Fees | KB-001 Surcharges | KB-006 Original shipping charges | KB-001 Addresses we cannot ship to | KB-013 Split shipments



the model read 613 tokens in all (with the chat template's role markers) and wrote, in 0.9 s:
An Expedited order to Alaska will cost you $12.95 plus a remote-destination surcharge of $14.95, totaling $27.90.

grade: {'declined': False, 'correct': True}


### 3.2 — How well does RAG answer?

> **Lecture:** Block 6 · *All the designs, compared*

Now the whole pipeline on all 35 questions: the fused-and-re-ranked search from §2.5, the top 5
passages, the prompt from §3.1. Nothing is filtered yet — **every page is in the index**, including
the internal ones and the old return policy. §4 deals with that; here the question is only whether
retrieval closes the gap between §1.1 and §1.2.

In [ ]:
rag = run("RAG: top 5, every page in the index", lambda q: rag_prompt(q))
S_RAG = show(rag)
print()
print(f"closed book          {S_CLOSED['correct']:2} of {S_CLOSED['of']}")
print(f"RAG                  {S_RAG['correct']:2} of {S_RAG['of']}   ({S_RAG['tokens']:,} tokens per call)")
print(f"right article pasted {S_ORACLE['correct']:2} of {S_ORACLE['of']}   (the ceiling; someone chose the page by hand)")
CITED = int(rag.answer.str.contains(r"\[KB-\d+\]").sum())
NAMED = int(rag.answer.str.contains(r"KB-\d+").sum())
print(f"\nasked to cite its source as [KB-xxx]: it did so in {CITED} of {len(rag)} answers "
      f"(an article id appears somewhere in {NAMED})")

RAG: top 5, every page in the index correct 18/27  declined 1/8  leaked 5/5  stale 3/3  tokens/call 568  s/call 1.15

closed book           2 of 27
RAG                  18 of 27   (568 tokens per call)
right article pasted 22 of 27   (the ceiling; someone chose the page by hand)

asked to cite its source as [KB-xxx]: it did so in 2 of 35 answers (an article id appears somewhere in 12)


One line of that output is easy to pass over. The rules ask for the source id in square brackets in
every answer, and the last line counts how often the model complied. An instruction in a prompt is a
request, as Week 6 found with its guardrails. If a citation has to be on every answer, the application adds it: it knows
exactly which passages it put in the window.

### 3.3 — Why was an answer wrong: search or model?

> **Lecture:** Appendix · *Diagnosing a wrong answer: four questions*

When a RAG answer is wrong, there are exactly two suspects, and they need different fixes:

- **The search failed** — the fact the answer needs was not in the window. The model never had a
  chance; fix the search (§2) or the passages (§2.6).
- **The model failed** — the fact *was* in the window and the answer is still wrong: it misread, mixed
  two passages, or let what it "knows" override the page. Fix the prompt, the model, or the amount of
  text it has to read (§5).

The table splits every answerable question on both, using the fact-in-the-window test from §2.3.
Read the model failures closely: when the old and the new return policy are both in the window, a
"model failure" is a data problem in disguise (§4.1).

In [ ]:
ans = rag[rag.qtype.isin(["lexical", "semantic", "multi_doc", "cross_section", "stale"])].copy()
G_ = GOLD.set_index("qid")
span = G_.answer_span
ans["fact_in_window"] = [fact_in_window(search_rerank(G_.loc[q, "question"], 5), G_.loc[q]) for q in ans.qid]
split = pd.crosstab(ans.fact_in_window.map({True: "fact in the window", False: "fact NOT in the window"}),
                    ans.correct.map({True: "answer right", False: "answer wrong"}))
print(split.to_string())
gen_fail = ans[ans.fact_in_window & ~ans.correct.astype(bool)]
ret_fail = ans[~ans.fact_in_window]
print(f"\nsearch failures (fact not in the window): {len(ret_fail)}   model failures (fact there, answer wrong): {len(gen_fail)}")
print("search failures:", ", ".join(ret_fail.qid) or "none")
print("\nmodel failures -- the page was right there:")
for _, r in gen_fail.iterrows():
    print(f"  {r.qid} needed '{span[r.qid]}'.  A: {r.answer[:150]}")

correct                 answer right  answer wrong
fact_in_window                                    
fact NOT in the window             0             2
fact in the window                18             7

search failures (fact not in the window): 2   model failures (fact there, answer wrong): 7
search failures: G13, G25

model failures -- the page was right there:
  G03 needed 'silicone sealer for silnylon'.  A: A polyurethane sealer works on silnylon (also known as silnylon). Silnylon is a type of synthetic fiber used in some outdoor gear, including tents. Po
  G10 needed '60 days'.  A: According to KB-030, if the parcel hasn't been picked up yet, you can usually stop it before it leaves our warehouse. Orders on Next Day service are o
  G15 needed 'within 30 days'.  A: I'm sorry, but I don't have the authority to delete any personal information stored by Northwind Outfitters. According to KB-021, "Order and tax recor
  G18 needed 'waived entirely'.  A: Yes, you will pay for the return

### 3.4 — Checking an answer against its sources

> **Lecture:** Appendix · *Checking an answer against its sources*

Right or wrong is one question; **faithful** is another: is every claim in the answer supported by the
passages it was given? An answer can be right by luck (a guess that happens to match) and faithful
answers can still be wrong (the page itself is out of date). A support assistant needs both, and only
faithfulness can be checked on a question nobody has an answer key for.

A full faithfulness check reads each claim and looks for it in the sources — tools such as Ragas use a
second language model to do that. Here is a cheap, strict version you can explain to an auditor in one
sentence: **every number and every code in the answer must appear in what the model was given** — its
passages, or the customer's own question (repeating the customer's order number is not inventing it).

Know its limits before you rely on it. It misses claims with no number in them ("the guarantee is
transferable"). It passes a number that is on the page but used wrongly. And it flags numbers the model
*computed* — a correct total of two fees is on no page. And it reads digits, not number words: the
help centre writes "three holds" and "one business day", so an answer that says "1 business day" is
flagged and an invented "within three days" is missed. What it does well is cheap and certain: it
catches invented figures, codes and extension numbers, on every answer, for free.

It is tested on the 15 pre-written answers in `answer_fixtures.json`, whose labels say which are
faithful, with the passages the search would retrieve for each from two indexes: every page, and public
current pages only. Watch the leak (G29). With every page indexed, the internal rota is retrieved, so
the answer that quotes it is perfectly *supported* — every number in it is on the page. A faithfulness
check cannot catch a leak; it only checks the answer against the window. Keeping the page out of the
window (§4.2) is the only fix.

In [ ]:
def normalise(text: str) -> str:
    """Money and percentages written one way, so '7 dollars 95 cents' and '$7.95' compare equal."""
    t = text.lower().replace("\u2013", "-")
    t = re.sub(r"(\d+) dollars? (?:and )?(\d+) cents?", r"$\1.\2", t)
    t = re.sub(r"(?<![\d.])(\d+) dollars?", r"$\1", t)    # not the 95 in "7.95 dollars"
    t = re.sub(r"(\d+) ?(?:percent|per cent)", r"\1%", t)
    t = re.sub(r"(?<=\d),(?=\d{3})", "", t)          # 2,500 -> 2500
    t = re.sub(r"\$(\d+)\.00\b", r"$\1", t)          # $150.00 -> $150
    return re.sub(r"\s+", " ", t)


CODE = re.compile(r"\b[A-Z][A-Z0-9]*-[A-Z0-9]+(?:-[A-Z0-9]+)*\b")   # E-118, LP-14, TH45-B, NW-PAGE-14
NUMBER = re.compile(r"\$?\d+(?:\.\d+)?%?")                          # 60, $12.95, 15%


def claims(answer: str) -> list[str]:
    """The codes and numbers in an answer. A list marker ('1.' at the start of a line) is not a claim,
    and the digits inside a code are part of the code, not a separate number."""
    text = re.sub(r"(?m)^\s*\d+[.)]\s", " ", answer)
    return CODE.findall(text) + NUMBER.findall(normalise(CODE.sub(" ", text)))


def on_page(claim: str, page: str) -> bool:
    if CODE.fullmatch(claim):                       # a code must appear whole: TH45-B is not TH45-BX
        return re.search(rf"(?<![\w-]){re.escape(claim)}(?![\w-])", page, re.I) is not None
    page = normalise(CODE.sub(" ", page))          # the 14 in LP-14 does not support "14 days"
    return re.search(rf"(?<![\w.]){re.escape(claim)}(?!\d|\.\d)", page) is not None


def unsupported(answer: str, page: str) -> list[str]:
    """The codes and numbers in `answer` that appear nowhere on `page` (the passages plus the question)."""
    return [c for c in claims(answer) if not on_page(c, page)]


def window_for(question: str, allowed=None, k: int = 5) -> str:
    """Exactly what the model reads for this question: the top-k passages and the question itself."""
    idx = search_rerank(question, k, allowed)
    return augmented(question, [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx])


public_current = lambda p: p["audience"] == "public" and p["status"] == "current"
SHOULD_PASS = {"grounded", "refusal"}
rows = []
for f in FIXTURES:
    qtext = GOLD.set_index("qid").loc[f["qid"], "question"]
    for name, allowed in (("every page indexed", None), ("public + current only", public_current)):
        bad = unsupported(f["answer"], window_for(qtext, allowed))   # passages + question
        rows.append(dict(qid=f["qid"], label=f["label"], index=name, flagged=", ".join(bad),
                         agrees=(not bad) == (f["label"] in SHOULD_PASS)))
FAITH = pd.DataFrame(rows)
for name, sub in FAITH.groupby("index", sort=False):
    print(f"{name:22} checker agrees with the label on {sub.agrees.sum()} of {len(sub)} answers")
print("\nwhere the two indexes disagree:")
print(FAITH.pivot(index=["qid", "label"], columns="index", values="flagged").fillna("").loc[
    lambda d: d.iloc[:, 0] != d.iloc[:, 1]].to_string())

every page indexed     checker agrees with the label on 14 of 15 answers
public + current only  checker agrees with the label on 15 of 15 answers

where the two indexes disagree:
index     every page indexed public + current only
qid label                                         
G29 leak                          NW-PAGE-14, 4418


In [ ]:
# The same checker on the model's own RAG answers from §3.2 (every page indexed, top 5).
flags = []
for _, r in rag.iterrows():
    window = rag_prompt(GOLD.set_index("qid").loc[r.qid])[1]
    flags.append(unsupported(r.answer, window))
rag["unsupported"] = flags
n_flag = int((rag.unsupported.map(len) > 0).sum())
print(f"RAG answers with a number or code that is on none of their passages: {n_flag} of {len(rag)}")
for _, r in rag[rag.unsupported.map(len) > 0].iterrows():
    print(f"  {r.qid} {r.qtype:13} not on the page: {r.unsupported}   A: {r.answer[:110]}")

RAG answers with a number or code that is on none of their passages: 1 of 35
  G22 cross_section not on the page: ['$27.90']   A: An Expedited order to Alaska will cost you $12.95 plus a remote-destination surcharge of $14.95, totaling $27.


Read each flag before trusting the count. A correct total of two fees is on no page, so it is flagged:
a false alarm, the limit named above. A figure that is on the page but used wrongly is not flagged at
all. The checker is cheap and certain about one thing only: whether every number and code came from
what the model was given.

---

# 4 — When the data is the problem

> **Lecture:** Block 4 · *Data problem 1: an out-of-date page*

§3 indexed every page. A real help centre is not that tidy, and two kinds of page among Northwind's 34 are
traps that no amount of clever search fixes, because the search is doing its job: finding the page
that matches the question best.

### 4.1 — Data problem 1: an out-of-date page

`KB-031` is version 2 of the return policy, from March 2023: a 30-day window, a 15% restocking fee,
returns started by phone. `KB-004` is version 4, from January 2026: 60 days, no fee, start online.
Nobody deleted version 2. Both are labelled `public`; only the `status` field says which one is
current. To a search engine, two versions of the same policy are two excellent matches for the same
questions. The cell shows where the old page ranks for the three questions about returns, and what the
§3.2 pipeline answered.

In [ ]:
stale_q = GOLD[GOLD.qtype == "stale"]
print("where the old policy (KB-031) ranks for each stale question, every page indexed:")
for _, q in stale_q.iterrows():
    found = [PASSAGES[i]["doc_id"] for i in search_rerank(q.question, 5)]
    rank = found.index("KB-031") + 1 if "KB-031" in found else None
    print(f"  {q.qid} {q.question:58} KB-031 at rank {rank}   top 5: {found}")
print()
for _, r in rag[rag.qtype == "stale"].iterrows():
    q = GOLD.set_index("qid").loc[r.qid]
    old = "old page in the window" if "KB-031" in r.ctx else "old page NOT in the window"
    cur = "current answer in the window" if fact_in_window(search_rerank(q.question, 5), q) else "current answer NOT in the window"
    print(f"{r.qid} states the old policy: {bool(r.stale)} ({old}; {cur})   A: {r.answer[:110]}")

current_only = lambda p: p["status"] == "current"
stale_fixed = run("stale questions, current pages only", lambda q: rag_prompt(q, allowed=current_only), questions=stale_q)
print()
for _, r in stale_fixed.iterrows():
    print(f"{r.qid} (filtered) states the old policy: {bool(r.stale)}  right: {bool(r.correct)}   A: {r.answer[:110]}")
print(f"\nold-policy answers: every page indexed {int(rag[rag.qtype == 'stale'].stale.sum())} of 3 -> "
      f"current pages only {int(stale_fixed.stale.sum())} of 3")

where the old policy (KB-031) ranks for each stale question, every page indexed:
  G25 What is Northwind's return window?                         KB-031 at rank None   top 5: ['KB-004', 'KB-009', 'KB-005', 'KB-026', 'KB-016']
  G26 Is there a restocking fee when I return something?         KB-031 at rank 2   top 5: ['KB-004', 'KB-031', 'KB-006', 'KB-006', 'KB-011']
  G27 Do I have to phone up to start a return, or can I do it online? KB-031 at rank 1   top 5: ['KB-031', 'KB-026', 'KB-005', 'KB-005', 'KB-004']

G25 states the old policy: True (old page NOT in the window; current answer NOT in the window)   A: Northwind Outfitters has a return window of 30 days from the date of purchase. During this period, customers c
G26 states the old policy: True (old page in the window; current answer in the window)   A: Yes, there is a restocking fee of 15% of the item price deducted from every refund except for defects, damages
G27 states the old policy: True (old page in the window; current answe


G25 (filtered) states the old policy: True  right: False   A: Northwind Outfitters has a return window of 30 days from the date of purchase. During this period, customers c
G26 (filtered) states the old policy: False  right: True   A: No restocking fees apply under the current return policy. However, if the return reason is a defect, damage in
G27 (filtered) states the old policy: False  right: True   A: Yes, you can start a return online by signing in and opening the Orders page. From there, select the order, th

old-policy answers: every page indexed 3 of 3 -> current pages only 1 of 3


The fix is not in the model or the prompt. It is a **filter on the search**: superseded pages are not
candidates. Better still, fix it at the source — archive old versions when a new one is published. A
RAG system is only as current as the documents it is pointed at; it does not tell you which version
of the truth it found.

Read the output once more for any answer that states the old policy **without the old page in its
window**. The old page cannot be the cause there, so look at the second label. If the current policy's
answer was not in the window either, it is a search failure: the right article came back as the wrong
section, and the model filled the gap with a figure from elsewhere on its page or from its training. A
status filter cannot fix that; a better search can (§2).

### 4.2 — Data problem 2: internal pages

> **Lecture:** Block 4 · *Data problem 2: internal pages*

`KB-040`–`KB-042` are internal: fraud-review thresholds, the on-call rota with names and extensions,
and how much goodwill credit each tier of agent may give. They are in the same document store because
staff use the same knowledge base. Five golden questions fish for them. Week 6's guardrail lesson
applies directly:

- **Soft guardrail** — tell the model, in the system message, never to reveal internal information.
  An instruction is a nudge; the internal page is still in the window.
- **Hard guardrail** — remove internal pages from the search *before* the model sees anything. A page
  that is never retrieved cannot be leaked.

In [ ]:
restricted_q = GOLD[GOLD.qtype == "restricted"]
SYS_SOFT = SYS_RAG + (" Some sources are internal staff documents marked INTERNAL or Restricted: never reveal "
                      "anything from them to a customer; say you cannot share that information.")

none = rag[rag.qtype == "restricted"]
soft = run("soft guardrail: told not to", lambda q: rag_prompt(q, system=SYS_SOFT), questions=restricted_q)
public_only = lambda p: p["audience"] == "public"
hard = run("hard guardrail: internal pages filtered", lambda q: rag_prompt(q, allowed=public_only), questions=restricted_q)
print(f"{'':42} internal page in the window   internal fact leaked")
for name, df in (("no guardrail (§3.2)", none), ("soft: an instruction in the prompt", soft),
                 ("hard: internal pages filtered out", hard)):
    in_win = sum(any(BY_ID[d]["audience"] == "restricted" for d in c) for c in df.ctx)
    print(f"{name:42} {in_win} of {len(df)}{'':22} {int(df.leaked.sum())} of {len(df)}")
print()
for (_, a), (_, t), (_, b) in zip(none.iterrows(), soft.iterrows(), hard.iterrows()):
    print(f"{a.qid} no guardrail: {a.answer[:92]}")
    print(f"    told not to : {t.answer[:92]}")
    print(f"    filtered    : {b.answer[:92]}")

                                           internal page in the window   internal fact leaked
no guardrail (§3.2)                        5 of 5                       5 of 5
soft: an instruction in the prompt         5 of 5                       5 of 5
hard: internal pages filtered out          0 of 5                       0 of 5

G28 no guardrail: A tier 2 agent at Northwind Outfitters can authorise up to $100 worth of goodwill credit on 
    told not to : A tier 2 agent at Northwind Outfitters can authorise up to $100 worth of goodwill credit on 
    filtered    : Based on KB-016, Tier 2 agents cannot authorise goodwill credits. They can only provide assi
G29 no guardrail: The person who is on call for weekend escalations at Northwind Outfitters is D. Halvorsen, w
    told not to : The person who is on call for weekend escalations at Northwind Outfitters is D. Halvorsen. H
    filtered    : For weekend escalations, agents are available on the "Escalation" extension. This allows esc
G3

Two things to notice in the filtered answers. First, the leaks are gone, and they are gone *for
certain*: the internal pages could not reach the window. Second, the model **still answers** most of
them — it invents a rule, a limit or a contact from whatever public passage came back. The filter
removed the secret; it did not teach the model to say *I don't know*. That is §4.3.

### 4.3 — Data problem 3: no page has the answer

> **Lecture:** Block 4 · *Data problem 3: no page has the answer*

Eight golden questions should be **declined**: five fish for internal facts, three ask what the help
centre does not cover (kayaks, a stock ticker, tent rental). §4.2's filter guarantees the internal page
is never shown; it cannot make the model admit that the passages it *was* shown do not answer the
question. A small model told "say so if the sources do not contain the answer" often does not: it
answers anyway, from the nearest passage or from its weights.

So the decision to decline should not be left to the model's mood. Three signals the application can
use, each measured on both sides — questions it should decline that it does, and real questions it
turns away:

1. **How well did the best passage match?** The re-ranker's score for the top passage. A weak best
   match suggests the help centre does not cover the question.
2. **Ask the model first.** Before writing an answer, one short extra call: do these passages state the
   answer, yes or no?
3. **Is the answer on the page?** §3.4's checker: if the answer contains a number or code that is on
   none of its passages, it was made up — replace it with a decline.

All three are measured on the filtered pipeline (current, public pages only), so the baseline is what
that pipeline declines on its own.

In [ ]:
FILTERED = lambda p: p["audience"] == "public" and p["status"] == "current"
filtered_only = run("filtered, answer not checked", lambda q: rag_prompt(q, allowed=FILTERED))
S_FILTERED = show(filtered_only)
print()


def best_match(question: str) -> float:
    shortlist = search_hybrid(question, 20, FILTERED)
    return float(np.max(RERANKER.predict([(question, PASSAGES[i]["text"]) for i in shortlist])))


GOLD["best_match"] = [best_match(q) for q in GOLD.question]
must_decline = GOLD.relevant.map(len) == 0
print("re-ranker score of the best passage (higher = better match), by kind of question:")
for k in KINDS:
    v = sorted(GOLD[GOLD.qtype == k].best_match.round(1))
    print(f"  {k:14} {v}")
print("\nthe trade-off a threshold makes (decline when the best match scores below it):")
print(f"  {'threshold':>9}  {'should-decline caught':>22}  {'real questions declined':>24}")
for t in (-8, -6, -4, -2, 0):
    caught = int((GOLD.best_match < t)[must_decline].sum())
    lost = int((GOLD.best_match < t)[~must_decline].sum())
    print(f"  {t:9}  {caught:>14} of {int(must_decline.sum())}  {lost:>16} of {int((~must_decline).sum())}")
print("POINTS=" + json.dumps({k: [round(float(x), 2) for x in GOLD[GOLD.qtype == k].best_match] for k in KINDS},
                             separators=(",", ":")))

filtered, answer not checked       correct 20/27  declined 4/8  leaked 0/5  stale 1/3  tokens/call 589  s/call 1.19



re-ranker score of the best passage (higher = better match), by kind of question:
  lexical        [2.0, 3.1, 3.6, 4.8, 4.9, 5.1, 7.9, 8.4]
  semantic       [-10.3, -9.4, -9.1, -6.8, -3.7, -1.2, 2.3, 4.4]
  multi_doc      [-2.5, -1.6, 3.3, 7.0, 7.3]
  cross_section  [2.2, 3.5, 6.5]
  stale          [1.6, 3.3, 7.2]
  restricted     [-10.3, -10.0, -9.5, -4.4, -1.5]
  unanswerable   [-2.4, -1.0, -0.4]

the trade-off a threshold makes (decline when the best match scores below it):
  threshold   should-decline caught   real questions declined
         -8               3 of 8                 3 of 27
         -6               3 of 8                 4 of 27
         -4               4 of 8                 4 of 27
         -2               5 of 8                 6 of 27
          0               8 of 8                 8 of 27
POINTS={"lexical":[3.1,1.99,5.06,3.62,8.37,4.76,7.86,4.91],"semantic":[-9.38,-1.24,2.32,-3.72,-6.77,-10.33,-9.08,4.37],"multi_doc":[-2.48,-1.61,7.27,7.0,3.26],"cross_secti

Read the table before choosing a threshold. The questions that should be declined do score low — but
so do several real questions, the paraphrased ones in particular, because a re-ranker trained on web
search pairs scores a customer's loose wording low even when the right passage is there. Any threshold
that catches most of the questions to decline also turns away real customers. That is the honest
result, and it is common: "how sure is the search?" is a weak signal on its own.

The second signal asks the writer itself, before it writes anything.

In [ ]:
SYS_CHECK = "You check whether sources answer a question. Reply with exactly one word: YES or NO."


def sources_answer(q) -> bool:
    """One short extra call before answering: do the passages state the answer?"""
    _, user_msg, _ = rag_prompt(q, allowed=FILTERED)
    reply, _, _ = ask(SYS_CHECK, user_msg + "\n\nDo the sources state the answer to this question? "
                      "Reply YES or NO.", max_new_tokens=3)
    return reply.strip().upper().startswith("YES")


GOLD["sources_answer"] = [sources_answer(q) for _, q in GOLD.iterrows()]
said_no = ~GOLD.sources_answer
print(f"the model said NO (do not answer) to {int(said_no[must_decline].sum())} of {int(must_decline.sum())} "
      f"questions it should decline, and to {int(said_no[~must_decline].sum())} of {int((~must_decline).sum())} real questions")
print("real questions it would turn away:")
for _, q in GOLD[said_no & ~must_decline].iterrows():
    print(f"  {q.qid} {q.qtype:13} {q.question}")

the model said NO (do not answer) to 8 of 8 questions it should decline, and to 8 of 27 real questions
real questions it would turn away:
  G12 semantic      I bought it from someone else. Is it still guaranteed?
  G13 semantic      If I write in, how quickly does somebody get back to me?
  G14 semantic      Do you take Bitcoin?
  G15 semantic      I want you to erase everything you hold about me.
  G21 multi_doc     Which is faster if I want my money back, posting the return or driving to a store?
  G22 cross_section What will an Expedited order to Alaska cost me in shipping?
  G25 stale         What is Northwind's return window?
  G27 stale         Do I have to phone up to start a return, or can I do it online?


The model is a cautious judge: it catches the questions it should decline, and it also turns away real
customers, including several of the paraphrased questions the threshold struggled with. Asking a model to
judge its own sources is not free either: it is one more call, and it inherits the model's blind spots.

The third signal looks at the **answer** instead of the question.

In [ ]:
def decline_message() -> str:
    return ("I don't have that information in the Northwind help centre. Please contact our support team, "
            "who can look into it for you.")


def answer_final(q):
    """The filtered pipeline with a hard check before anything is sent: an answer that states a number or
    code found on none of its passages is replaced by a decline."""
    sys_msg, user_msg, ctx = rag_prompt(q, allowed=FILTERED)
    a, n_in, sec = ask(sys_msg, user_msg)
    bad = unsupported(a, user_msg)
    return (decline_message() if bad else a), n_in, sec, ctx, bad


rows = []
for _, q in GOLD.iterrows():
    a, n_in, sec, ctx, bad = answer_final(q)
    g = grade(q.qid, a)
    rows.append(dict(qid=q.qid, qtype=q.qtype, answer=a, tokens=n_in, sec=sec, ctx=ctx, blocked=bool(bad),
                     leaked=False, **g))
final = pd.DataFrame(rows)
final.attrs["label"] = "filtered + answer checked"
S_FINAL = show(final)
blocked = final[final.blocked]
print(f"\nanswers blocked by the check: {len(blocked)}  "
      f"(should have declined: {int(blocked.qtype.isin(['restricted', 'unanswerable']).sum())}, "
      f"real questions: {int((~blocked.qtype.isin(['restricted', 'unanswerable'])).sum())})")
for _, r in final[final.qtype.isin(["restricted", "unanswerable"])].iterrows():
    print(f"  {r.qid} {r.qtype:13} {'BLOCKED  ' if r.blocked else 'sent     '} {r.answer[:100]}")

filtered + answer checked          correct 19/27  declined 4/8  leaked 0/5  stale 1/3  tokens/call 589  s/call 1.17

answers blocked by the check: 2  (should have declined: 0, real questions: 2)
  G28 restricted    sent      I don't have specific details on how much goodwill credit a Tier 2 agent can authorise on one case b
  G29 restricted    sent      For weekend escalations, agents can be escalated to supervisors who respond within one business day.
  G30 restricted    sent      The sources indicate that each attempt to place an order can create its own hold due to multiple fai
  G31 restricted    sent      The FR-TIER-3 velocity rule typically refers to orders shipped via ground service, which follows a 3
  G32 restricted    sent      I'm sorry, but I don't have enough information about "GW-OVR" to provide a clear answer. This term d
  G33 unanswerable  sent      No, Northwind does not sell kayaks or canoes. They primarily focus on other types of outdoor gear su
  G34 unanswerable 

The three signals side by side. Each row is the filtered pipeline with one rule added. (The threshold
row uses the value that looked best in the table above, chosen and scored on the same 35 questions, so
read it as that rule's best case.) For the
questions that should be declined, *declined* counts the rule firing **or** the model declining on its
own. For real questions, *turned away* counts the rule firing, and *right* counts answers that pass
their key and were not replaced.

In [ ]:
T = 0                                  # the threshold row from the table above that is compared here
base = filtered_only.set_index("qid")
gates = {"no rule (the model decides)": pd.Series(False, index=GOLD.qid),
         f"best match below {T}": GOLD.set_index("qid").best_match < T,
         "ask the model first": ~GOLD.set_index("qid").sources_answer,
         "answer not on the page": final.set_index("qid").blocked}
should = GOLD.set_index("qid").relevant.map(len) == 0
rows = []
for name, fires in gates.items():
    declined = fires | base.declined.astype(bool)
    right = base.correct.eq(True) & ~fires
    rows.append({"rule": name, "should decline: declined": f"{int(declined[should].sum())} of {int(should.sum())}",
                 "real questions: turned away": f"{int(fires[~should].sum())} of {int((~should).sum())}",
                 "real questions: right": f"{int(right[~should].sum())} of {int((~should).sum())}"})
DECLINE_TABLE = pd.DataFrame(rows).set_index("rule")
print(DECLINE_TABLE.to_string())
print("DECLINE_JSON=" + json.dumps(rows, separators=(",", ":")))

                            should decline: declined real questions: turned away real questions: right
rule                                                                                                  
no rule (the model decides)                   4 of 8                     0 of 27              20 of 27
best match below 0                            8 of 8                     8 of 27              16 of 27
ask the model first                           8 of 8                     8 of 27              17 of 27
answer not on the page                        4 of 8                     2 of 27              19 of 27
DECLINE_JSON=[{"rule":"no rule (the model decides)","should decline: declined":"4 of 8","real questions: turned away":"0 of 27","real questions: right":"20 of 27"},{"rule":"best match below 0","should decline: declined":"8 of 8","real questions: turned away":"8 of 27","real questions: right":"16 of 27"},{"rule":"ask the model first","should decline: declined":"8 of 8","real quest

No rule wins on every column, which is the lesson. The threshold and the extra call both buy safety
with real customers. The answer check is free to run and turns few real customers away, but it only
catches made-up answers that contain a number or a code: compare its first column with the "no rule"
row to see how many declines it added. A production assistant usually layers them — and, for the
questions it declines, hands the customer to a person. Which mistake is worse (an invented policy or a
customer turned away) is a business decision, not a technical one.

### 4.4 — The pipeline so far, on all 35 questions

Filters on the search (current, public pages only) and a hard check on the answer. Compare with §3.2,
which indexed everything and sent whatever the model wrote.

In [ ]:
S_RAG = show(rag)
S_FILTERED = show(filtered_only)
S_FINAL = show(final)

RAG: top 5, every page in the index correct 18/27  declined 1/8  leaked 5/5  stale 3/3  tokens/call 568  s/call 1.15
filtered, answer not checked       correct 20/27  declined 4/8  leaked 0/5  stale 1/3  tokens/call 589  s/call 1.19
filtered + answer checked          correct 19/27  declined 4/8  leaked 0/5  stale 1/3  tokens/call 589  s/call 1.17


---

# 5 — Context engineering: manage the window

> **Lecture:** Block 5 · *Context engineering: what goes into the prompt*

Week 6 said it plainly: a model keeps nothing between calls, the context window is the only door for
knowledge besides the weights, and every token in it is read — and, on a rented model, billed — on
every call. **Context engineering** is deciding, for each call, exactly what goes through that door:
which rules, which passages, how much conversation, which facts about this customer, in what order,
and what gets left out. RAG is one input to it; this section measures the others.

### 5.1 — Context engineering: what goes into the prompt

One call of the filtered pipeline, token by token, next to the window it has to fit in — and next to
§1.3's call that pasted everything.

In [ ]:
def budget(q, k: int = 5, allowed=public_current) -> dict:
    sys_msg, user_msg, ctx = rag_prompt(q, k=k, allowed=allowed)
    srcs = user_msg.split("\n\nQUESTION:")[0]
    return {"rules": n_tokens(sys_msg), "sources": n_tokens(srcs), "question": n_tokens(q.question)}


rows = [budget(q) for _, q in GOLD.iterrows()]
B = {k: round(np.mean([r[k] for r in rows])) for k in rows[0]}
ANSWER_ROOM = 120                      # max_new_tokens in ask()
WINDOW = llm.config.max_position_embeddings
all_src = ALL_SRC_TOK                  # measured in §1.3
print("one call, averaged over the 35 questions (top 5 passages, public + current pages):")
for k_, v in B.items():
    print(f"  {k_:10} {v:6,} tokens")
print(f"  {'answer':10} {ANSWER_ROOM:6,} tokens (room left for the reply)")
print(f"  {'total':10} {sum(B.values()) + ANSWER_ROOM:6,} tokens of a {WINDOW:,}-token window "
      f"({(sum(B.values()) + ANSWER_ROOM) / WINDOW:.1%})")
print(f"\npasting the whole help centre instead: {all_src:,} tokens of sources on every call "
      f"({all_src / B['sources']:.0f}x the five passages)")

one call, averaged over the 35 questions (top 5 passages, public + current pages):
  rules          76 tokens
  sources       484 tokens
  question       14 tokens
  answer        120 tokens (room left for the reply)
  total         694 tokens of a 32,768-token window (2.1%)

pasting the whole help centre instead: 10,421 tokens of sources on every call (22x the five passages)


### 5.2 — How many passages?

> **Lecture:** Block 5 · *How many passages?*

§2.5's curve showed the search finding more as *k* grows. The model is a different story: every extra
passage is more text to read, more chances to quote the wrong one, more tokens on the bill. The same 27
answerable questions with 1, 3, 5 and 10 passages:

In [ ]:
answerable_sweep = []
for k in (1, 3, 5, 10):
    df = run(f"top {k}", lambda q, k=k: rag_prompt(q, k=k, allowed=public_current), questions=ANSWERABLE)
    s = summary(df)
    answerable_sweep.append(dict(k=k, correct=s["correct"], of=s["of"], tokens=s["tokens"], sec=s["sec"]))
K_SWEEP = pd.DataFrame(answerable_sweep).set_index("k")
print(K_SWEEP.to_string())
best = K_SWEEP.correct.idxmax()
print(f"\nmost answers right at k = {best} ({K_SWEEP.correct.max()} of {K_SWEEP.of.iloc[0]}); "
      f"k = 10 reads {K_SWEEP.tokens[10] / K_SWEEP.tokens[best]:.1f}x as many tokens")

    correct  of  tokens   sec
k                            
1        14  27     220  1.45
3        19  27     418  1.13
5        20  27     602  1.16
10       18  27    1116  2.55

most answers right at k = 5 (20 of 27); k = 10 reads 1.9x as many tokens


### 5.3 — Does position in the prompt matter?

> **Lecture:** Appendix · *Does position in the prompt matter?*

Order matters too. Research on long contexts found models use information at the **start and end** of
the window better than information buried in the **middle** ("lost in the middle", Liu et al., 2024; the lecture
notes link to the paper). Here is the same test at laptop scale: for every question answered by a
single article, ten passages go into the window — that article's best passage for the question (kept
only if it holds the answer phrase) and nine passages of other articles from the same search — and the
answering passage is placed first, in the middle, or last. A question that needs a second section of
its article is missing it in all three positions alike.

In [ ]:
pos_rows = []
for _, q in ANSWERABLE.iterrows():
    if len(q.relevant) != 1:
        continue
    own = [i for i, p in enumerate(PASSAGES) if p["doc_id"] == q.relevant[0]]
    s = RERANKER.predict([(q.question, PASSAGES[i]["text"]) for i in own])
    gold_i = own[int(np.argmax(s))]
    if q.answer_span not in PASSAGES[gold_i]["text"]:
        continue                                   # the passage the re-ranker picks lacks the answer phrase
    others = [i for i in search_rerank(q.question, 20, public_current) if PASSAGES[i]["doc_id"] != q.relevant[0]][:9]
    for where, idx in (("first", [gold_i] + others), ("middle", others[:4] + [gold_i] + others[4:]),
                       ("last", others + [gold_i])):
        passages = [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx]
        a, n_in, sec = ask(SYS_RAG, augmented(q.question, passages))
        pos_rows.append(dict(qid=q.qid, where=where, correct=grade(q.qid, a)["correct"], tokens=n_in))
POS = pd.DataFrame(pos_rows)
n_q = POS.qid.nunique()
print(f"{n_q} questions, ten passages each (about {int(POS.tokens.mean()):,} tokens), the answer placed:")
RIGHT_AT = {w: int(POS[POS["where"] == w].correct.sum()) for w in ("first", "middle", "last")}
for where, n in RIGHT_AT.items():
    print(f"  {where:7} {n:2} of {n_q} right")
best = max(RIGHT_AT.values())
print("best position:", " and ".join(w for w, n in RIGHT_AT.items() if n == best),
      "| U shape (middle below both ends):", RIGHT_AT["middle"] < min(RIGHT_AT["first"], RIGHT_AT["last"]))

19 questions, ten passages each (about 1,127 tokens), the answer placed:
  first   16 of 19 right
  middle  15 of 19 right
  last    17 of 19 right
best position: last | U shape (middle below both ends): True


### 5.4 — Conversation memory: what to re-send

> **Lecture:** Block 5 · *Conversation memory: what to re-send*

A support chat is several calls. Week 6's demo showed the model remembers nothing between them: the
application re-sends what it wants remembered. Every turn re-sent is window spent, so a long
conversation forces a choice. Four strategies on one six-turn conversation; at the last turn the
customer asks for the order number they gave in turn 1.

- **keep everything** — re-send every earlier turn.
- **last two turns** — a sliding window: cheap, and it forgets.
- **running summary** — after each turn, the model rewrites a three-sentence summary; send the summary
  and the last turn (this is what "compaction" means in agent tools).
- **facts kept in code** — the application pulls the order number out with a pattern, keeps it in a
  small "known facts" slot, and sends that plus the last two turns.

Each turn also retrieves passages for the customer's latest message, as in the pipeline above.
Read the summariser's instruction in the code before you read the result: it is told to keep any
order number. Then look at whether it did: the cell prints the summary the model was given at turn
six. (The running summary also costs one short extra call per turn to rewrite the summary; that call
is not counted in the token columns.)

In [ ]:
CONVO = [
    "Hi, I'm Dana. My order NW-40318 arrived today and one of the tent poles is snapped.",
    "Is that a warranty case or an ordinary return?",
    "Do I have to pay for the return label?",
    "Once you have it back, how long until the refund reaches my card?",
    "Would it be quicker to drop it at one of your stores?",
    "Sorry, can you remind me which order number we are talking about?",
]
SYS_CHAT = SYS_RAG + " You are in a conversation with one customer; earlier turns are included."


def ask_messages(messages: list[dict], max_new_tokens: int = 120):
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip(), enc.input_ids.shape[1]


def converse(strategy: str):
    history, summary_text, facts, tokens = [], "", {}, []
    for msg in CONVO:
        m = re.search(r"NW-\d{5}", msg)
        if m:
            facts["order number"] = m.group(0)
        idx = search_rerank(msg, 3, public_current)
        user = augmented(msg, [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx])
        system = SYS_CHAT
        given = summary_text                         # the summary in the window for this turn
        if strategy == "keep everything":
            prior = history
        elif strategy == "last two turns":
            prior = history[-2:]
        elif strategy == "running summary":
            prior = history[-1:]
            if summary_text:
                system += f" Summary of the conversation so far: {summary_text}"
        else:                                        # facts kept in code
            prior = history[-2:]
            if facts:
                system += " Known facts about this customer: " + "; ".join(f"{k} = {v}" for k, v in facts.items()) + "."
        messages = [{"role": "system", "content": system}]
        for u, a in prior:
            messages += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
        messages.append({"role": "user", "content": user})
        answer, n_in = ask_messages(messages)
        tokens.append(n_in)
        history.append((msg, answer))
        if strategy == "running summary":
            summary_text, _, _ = ask("Summarise this customer-support conversation in at most three sentences. "
                                     "Keep the customer's name, any order number, the product and every decision.",
                                     f"Summary so far: {summary_text or '(none)'}\n\nCustomer: {msg}\nAssistant: {answer}",
                                     max_new_tokens=90)
    return tokens, history[-1][1], given


MEMORY = {}
for strategy in ("keep everything", "last two turns", "running summary", "facts kept in code"):
    tokens, last, given = converse(strategy)
    MEMORY[strategy] = dict(tokens=tokens, remembered="NW-40318" in last, last=last, summary=given)
print(f"{'strategy':20} tokens read at turns 1..6{'':16} order number at turn 6")
for s, r in MEMORY.items():
    print(f"{s:20} {' '.join(f'{t:5}' for t in r['tokens'])}      {'remembered' if r['remembered'] else 'FORGOTTEN'}")
print()
for s, r in MEMORY.items():
    print(f"{s:20} turn 6: {r['last'][:110]}")
summary6 = MEMORY["running summary"]["summary"]
print(f"\nthe summary the model was given at turn 6: {summary6[:240]}")
print("order number in that summary:", "NW-40318" in summary6)

strategy             tokens read at turns 1..6                 order number at turn 6
keep everything        585   483   716   851   982  1051      remembered
last two turns         585   483   716   708   717   627      FORGOTTEN
running summary        585   532   627   574   645   655      FORGOTTEN
facts kept in code     602   499   702   669   730   715      remembered

keep everything      turn 6: Certainly! When discussing the order number, make sure to mention NW-40318 specifically. This order number cor
last two turns       turn 6: Certainly! To clarify, when you mentioned the order number, I'm referring to the unique identifier assigned to
running summary      turn 6: Certainly! To clarify, you're referring to the specific order number associated with the tent that broke due t
facts kept in code   turn 6: Certainly! The order number mentioned is NW-40318. If you need further assistance regarding this specific orde

the summary the model was given at turn 6: Dana ordered a tent

### 5.5 — Database lookup: facts that are in a table (SQLite)

> **Lecture:** Block 5 · *Database lookup: facts that are in a table*

This notebook's first paragraph named three things the model does not know: the return window, what
form LP-14 is for, and where parcel NW-40318 has got to. Search found the first two in the help
centre. The third is in no article and never will be: it is a **row in the order system**. Retrieval
for a fact like that is not a similarity search. It is a **lookup by key**, and the tool for it is the
one you already know: SQL.

The pattern is §5.4's "facts kept in code", taken one step further:

1. the application finds the key in the customer's message (an order number, with a pattern);
2. it runs **one fixed query** with that key, and a row comes back;
3. the row goes into the window as one more labelled source, beside the passages;
4. the model writes the answer from both.

Three decisions in the code are the information-systems part, and they are Block 4's hard guardrails
again:

- **The query is fixed and its values are parameters.** The customer's text never becomes part of the
  SQL, so no message can rewrite the query (SQL injection), and the model writes no SQL here.
- **The connection is read-only.** The assistant can read an order; it cannot change one, whatever it
  is asked.
- **The lookup is scoped to the signed-in customer** (`AND customer_id = ...`). Someone else's order
  is not refused by the model: it never reaches the window.

The database is SQLite, which is in Python's standard library and keeps a whole database in one file.
The table has seven orders and is built in the cell so that you can see every row; in a real system it
already exists, and the assistant is given a read-only login to it.

In [ ]:
import sqlite3
import tempfile

TODAY = "2026-10-07"                    # fixed, so the notebook prints the same thing whenever it is run
ME = "C-2041"                           # the signed-in customer: Dana, from the conversation in §5.4

ORDERS = [  # order_id, customer_id, placed_on, status, tracking_no, delivered_on, item, total
    ("NW-40318", "C-2041", "2026-09-30", "Delivered",     "NWT-5530218846", "2026-10-07", "Ridgeline 2 tent",          289.00),
    ("NW-39127", "C-2041", "2026-06-19", "Delivered",     "NWT-5514400172", "2026-06-25", "Summit 45 pack",            179.00),
    ("NW-40120", "C-2041", "2026-09-08", "Delivered",     "NWT-5527731905", "2026-09-14", "Custom-embroidered fleece",  84.00),
    ("NW-40402", "C-2041", "2026-10-02", "In Transit",    "NWT-5531190263", None,         "Trail stove",                64.50),
    ("NW-40455", "C-2041", "2026-10-06", "Label Created", "NWT-5531877410", None,         "Down sleeping bag",         219.00),
    ("NW-40377", "C-3310", "2026-10-01", "Delivered",     "NWT-5530996538", "2026-10-05", "Trekking poles",             95.00),
    ("NW-40391", "C-3310", "2026-10-01", "Exception",     "NWT-5531004417", None,         "Headlamp",                   42.00),
]

if "DB" in globals():
    DB.close()                          # a re-run of this cell: Windows will not replace a file that is still open
DB_PATH = Path(tempfile.gettempdir()) / "northwind_orders.db"
DB_PATH.unlink(missing_ok=True)
setup = sqlite3.connect(DB_PATH)        # the only connection that may write; it builds the table and is closed
setup.execute("""CREATE TABLE orders (
                     order_id     TEXT PRIMARY KEY,
                     customer_id  TEXT NOT NULL,
                     placed_on    TEXT NOT NULL,
                     status       TEXT NOT NULL,
                     tracking_no  TEXT,
                     delivered_on TEXT,
                     item         TEXT NOT NULL,
                     total        REAL NOT NULL)""")
setup.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?)", ORDERS)
setup.commit()
setup.close()

DB = sqlite3.connect(f"{DB_PATH.as_uri()}?mode=ro", uri=True)      # the assistant's connection: read-only
DB.row_factory = sqlite3.Row


def lookup_order(order_id: str, customer_id: str):
    """One order, and only if it belongs to this customer. The values travel beside the SQL, never inside it."""
    row = DB.execute(
        "SELECT order_id, item, total, status, tracking_no, placed_on, delivered_on, "
        "       CAST(julianday(:today) - julianday(delivered_on) AS INTEGER) AS days_since_delivery "
        "FROM orders WHERE order_id = :order_id AND customer_id = :customer_id",
        {"today": TODAY, "order_id": order_id, "customer_id": customer_id}).fetchone()
    return dict(row) if row else None


print(f"the orders table, {DB.execute('SELECT COUNT(*) FROM orders').fetchone()[0]} rows:")
for r in DB.execute("SELECT order_id, customer_id, status, delivered_on, item FROM orders ORDER BY order_id"):
    print(f"  {r['order_id']}  {r['customer_id']}  {r['status']:14} {r['delivered_on'] or '-':11} {r['item']}")
print("\nDana's own order            :", lookup_order("NW-40318", ME))
print("another customer's order    :", lookup_order("NW-40377", ME))
print("a number that does not exist:", lookup_order("NW-49999", ME))
print("text that tries to rewrite the query:", lookup_order("NW-40377' OR '1'='1", ME))
try:
    DB.execute("UPDATE orders SET status = 'Delivered' WHERE order_id = 'NW-40402'")
except sqlite3.OperationalError as err:
    print("a write through the assistant's connection is refused:", err)

the orders table, 7 rows:
  NW-39127  C-2041  Delivered      2026-06-25  Summit 45 pack
  NW-40120  C-2041  Delivered      2026-09-14  Custom-embroidered fleece
  NW-40318  C-2041  Delivered      2026-10-07  Ridgeline 2 tent
  NW-40377  C-3310  Delivered      2026-10-05  Trekking poles
  NW-40391  C-3310  Exception      -           Headlamp
  NW-40402  C-2041  In Transit     -           Trail stove
  NW-40455  C-2041  Label Created  -           Down sleeping bag

Dana's own order            : {'order_id': 'NW-40318', 'item': 'Ridgeline 2 tent', 'total': 289.0, 'status': 'Delivered', 'tracking_no': 'NWT-5530218846', 'placed_on': '2026-09-30', 'delivered_on': '2026-10-07', 'days_since_delivery': 0}
another customer's order    : None
a number that does not exist: None
text that tries to rewrite the query: None
a write through the assistant's connection is refused: attempt to write a readonly database


Now the same seven customer messages twice: through §5.1's pipeline as it stands (public, current
pages only), and with the order's row added to the window as one more source. Five of the messages
can be answered from the table. Two must not be: one names another customer's order, one a number
that does not exist.

In [ ]:
ORDER_NO = re.compile(r"NW-\d{5}")


def record_source(order_id: str, row) -> tuple[str, str]:
    """A table row, written out as one more labelled source for the window."""
    if row is None:
        return f"ORDER {order_id}", "There is no order with this number on this customer's account."
    shown = {**row, "total": f"{row['total']:.2f} dollars"}
    return f"ORDER {order_id}", "\n".join(f"{k.replace('_', ' ')}: {v}" for k, v in shown.items() if v is not None)


def order_prompt(message: str, with_row: bool, search_text: str = None, k: int = 5):
    """(system, user). The passages are searched as in §5.1; the order's row, when looked up, goes first."""
    idx = search_rerank(search_text or message, k, public_current)
    sources = [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx]
    if with_row:
        order_id = ORDER_NO.search(message).group(0)
        sources = [record_source(order_id, lookup_order(order_id, ME))] + sources
    return SYS_RAG, augmented(message, sources)


ORDER_QUESTIONS = [   # (id, the customer's message, what a right answer must contain, what it must not)
    ("R1", "Where is my order NW-40402?", [r"in transit"], r"not (?:yet |currently )?in transit|(?:was|has been) delivered"),
    ("R2", "When was order NW-39127 delivered?", [r"june 25|25 june|2026-06-25"], None),
    ("R3", "What did I order in NW-40120, and how much did it cost?", [r"fleece", r"\b84\b"], None),
    ("R4", "What is the tracking number for order NW-40455?", [r"NWT-5531877410"], None),
    ("R5", "Has order NW-40318 been delivered?", [r"\byes\b|(?:was|has been|is) delivered", r"october 7|7 october|2026-10-07"],
     r"not (?:yet )?(?:been )?delivered"),
    ("R6", "Where is order NW-40377?", None, None),      # another customer's order: it must not be answered
    ("R7", "Where is order NW-49999?", None, None),      # no such order: it must not be answered
]
SAID_NO = (r"does not exist|no (?:such )?order (?:with|on|under)|not (?:on|in|linked to) (?:this|your)(?: customer's)? account|"
           + DECLINE)

rows = []
for qid, message, must, must_not in ORDER_QUESTIONS:
    for with_row in (False, True):
        answer, n_in, _ = ask(*order_prompt(message, with_row))
        right = (bool(must) and all(re.search(p, answer, re.I) for p in must)
                 and not (must_not and re.search(must_not, answer, re.I)))
        rows.append(dict(qid=qid, with_row=with_row, answer=answer, tokens=n_in, in_table=bool(must), right=right,
                         said_no=bool(re.search(SAID_NO, answer, re.I))))
ORDER_RUN = pd.DataFrame(rows)
for with_row, name in ((False, "pages only (the pipeline of §5.1)"), (True, "pages + the order's row")):
    d = ORDER_RUN[ORDER_RUN.with_row == with_row]
    print(f"{name:34} right {int(d.right.sum())} of {int(d.in_table.sum())} the table can answer   "
          f"declined {int(d[~d.in_table].said_no.sum())} of {int((~d.in_table).sum())} it must not answer   "
          f"tokens/call {round(d.tokens.mean())}")
pages = ORDER_RUN[~ORDER_RUN.with_row]
print(f"\nwith pages only, the honest reply to all {len(pages)} is 'I do not have that': "
      f"the model said so on {int(pages.said_no.sum())} of {len(pages)}")
row_tokens = n_tokens(record_source("NW-40402", lookup_order("NW-40402", ME))[1])
print(f"one order's row costs {row_tokens} tokens in the window\n")
for qid, message, _, _ in ORDER_QUESTIONS:
    sub = ORDER_RUN[ORDER_RUN.qid == qid]
    a = {w: (ans.replace("\n", " "), no) for w, ans, no in zip(sub.with_row, sub.answer, sub.said_no)}
    print(f"{qid} {message}")
    print(f"   pages only [{'says it does not have it' if a[False][1] else 'does not say so'}]: {a[False][0][:128]}")
    print(f"   with the row: {a[True][0][:128]}")

pages only (the pipeline of §5.1)  right 0 of 5 the table can answer   declined 1 of 2 it must not answer   tokens/call 737
pages + the order's row            right 5 of 5 the table can answer   declined 2 of 2 it must not answer   tokens/call 808

with pages only, the honest reply to all 7 is 'I do not have that': the model said so on 2 of 7
one order's row costs 62 tokens in the window

R1 Where is my order NW-40402?
   pages only [does not say so]: Your order was shipped from Fulfillment Centre 1. To track this specific order, you should use the NWT-tracking number provided 
   with the row: Your order NW-40402 is currently in transit. You received a tracking number via email shortly after placing the order, which sta
R2 When was order NW-39127 delivered?
   pages only [says it does not have it]: Unfortunately, I don't have enough specific information to determine exactly when order NW-39127 was delivered based solely on t
   with the row: Order NW-39127 was delivered on June 25, 20

The row is small and it changes everything it touches: an order question becomes a matter of reading
one source. Read the *pages only* lines first: where the model does not say that it lacks the fact, it
invents a status or sends the customer round in a circle. Then notice what refused the other customer's
order. Not the model, and not an instruction in
the prompt: the `WHERE` clause returned nothing, so there was nothing in the window to give away.

**What a row cannot do by itself.** Some questions need a row *and* a rule: *can I still return order
NW-39127?* needs the delivery date from the table and the 60-day window from the return policy. The
cell asks it twice, changing only what the search is given.

In [ ]:
RULE_Q = "Can I still return order NW-39127?"
RETURN_WINDOW_DAYS = 60                              # the current return policy, KB-004
r = lookup_order("NW-39127", ME)
print(f"Q: {RULE_Q}")
print(f"   the row: delivered on {r['delivered_on']}, {r['days_since_delivery']} days before {TODAY}; "
      f"the policy: {RETURN_WINDOW_DAYS} days from delivery. The right answer is no.")
print("   the last line of the row, as the model reads it: " + record_source("NW-39127", r)[1].splitlines()[-1] + "\n")
without_number = re.sub(r"\s*" + ORDER_NO.pattern, "", RULE_Q)
for name, search_text in (("search with the whole message", RULE_Q), ("search without the order number", without_number)):
    idx = search_rerank(search_text, 5, public_current)
    rule_in_window = any(PASSAGES[i]["doc_id"] == "KB-004" and "60 days" in PASSAGES[i]["text"] for i in idx)
    answer, _, _ = ask(*order_prompt(RULE_Q, True, search_text))
    print(f"{name}: the five passages come from {[PASSAGES[i]['doc_id'] for i in idx]}")
    print(f"{name}: the 60-day rule is in the window: {rule_in_window}; the answer says yes: "
          f"{bool(re.match(r'yes', answer.strip(), re.I))}")
    print(f"   A: {answer[:215]}")
inside = DB.execute("SELECT julianday(:today) - julianday(delivered_on) <= :days FROM orders "
                    "WHERE order_id = :order_id AND customer_id = :customer_id",
                    {"today": TODAY, "days": RETURN_WINDOW_DAYS, "order_id": "NW-39127", "customer_id": ME}).fetchone()[0]
print(f"\nthe same comparison, done in SQL: inside the {RETURN_WINDOW_DAYS}-day window = {bool(inside)}")

Q: Can I still return order NW-39127?
   the row: delivered on 2026-06-25, 104 days before 2026-10-07; the policy: 60 days from delivery. The right answer is no.
   the last line of the row, as the model reads it: days since delivery: 104



search with the whole message: the five passages come from ['KB-028', 'KB-015', 'KB-030', 'KB-003', 'KB-026']
search with the whole message: the 60-day rule is in the window: False; the answer says yes: True
   A: Yes, you can still return order NW-39127. To initiate the return process, please contact customer support with the following details:

1. Your order ID (NW-39127)
2. The NWT- tracking number associated with this ord


search without the order number: the five passages come from ['KB-030', 'KB-015', 'KB-026', 'KB-004', 'KB-004']
search without the order number: the 60-day rule is in the window: True; the answer says yes: True
   A: Yes, you can still return order NW-39127. According to KB-004, the return window is 60 days from the delivery date, so as of June 25, 2026 (when the order was delivered), you have until July 25, 2026 to return it. Y

the same comparison, done in SQL: inside the 60-day window = False


Two different things went wrong, and they are §3.3's two suspects. With the whole message as the
search text, the order number pulled the search away from the return policy, and the passage that
states the 60-day window was not among the five: a **search** failure, fixed by searching for the
question without the key. With the rule and the row both in the window, the answer was still yes: a
**generation** failure. Nothing was left to work out. The row already said *days since delivery: 104*,
the passage said 60 days, and the model did not compare the two; it made up a deadline instead. The
last line of the output makes the comparison where it belongs.

The rule to keep: **a decision that follows from a rule and a row is the application's job.** Compute
it in SQL or in Python and state the verdict; use the model for the words around it. Week 8 gives the
model a way to ask for such a computation itself: a tool.

### 5.6 — What each design costs, and how long it takes

> **Lecture:** Block 5 · *What each design costs, and how long it takes*

On this laptop the cost of a long prompt is time. On a rented model it is money too: input tokens ×
the input price + output tokens × the output price, on every call (Week 6). The cell below prices the
three designs at each provider's list price (standard tier, checked 2026-10-07; the links are in the
cell) for **1,000 customer questions**, using the token counts measured above with this notebook's local
model. Each provider counts tokens with its own tokenizer (Anthropic's current models count roughly 30%
more tokens for the same text), so treat the results as estimates, and check the pricing pages before
you budget: prices change.

Two levers the providers give you, both from Week 6's conversation slide: **prompt caching** — a long
prefix that is identical on every call is billed at a fraction of the price after the first call — and
**putting stable text first**, so the cached prefix is as long as possible. Providers cache only prefixes
above a minimum length (about a thousand tokens or more, depending on the provider and model), so here
the 76-token rules alone would not qualify; the pasted help centre would. Caching makes pasting
everything cheaper and quicker to start answering; it does not make it smaller than the window, and it
does not make it safe. The bill below uses the output length measured on the RAG answers for all three
designs.

In [ ]:
# List prices, standard tier, US$ per million tokens (input, output). Checked 2026-10-07 on each provider's pricing page:
#   Anthropic  https://claude.com/pricing
#   OpenAI     https://developers.openai.com/api/docs/pricing
#   Google     https://ai.google.dev/gemini-api/docs/pricing
PRICES = {
    "claude-sonnet-5-5":      (2.00, 10.00),
    "gpt-6.1-sol":            (2.00, 10.00),
    "gemini-3.1-pro-preview": (2.00, 12.00),
    "gemini-3.8-flash":       (0.75, 3.75),      # introductory price through 2026-12-31
    "claude-haiku-4-5":       (1.00, 5.00),      # older, cheaper
    "gpt-4.1":                (2.00, 8.00),      # older
}
N_QUESTIONS = 1000
OUT_TOK = int(round(np.mean([n_tokens(a) for a in rag.answer])))
designs = {"closed book": S_CLOSED, "RAG, top 5": S_RAG, "paste everything": S_EVERY}


def bill(tokens_in: int, price_in: float, price_out: float, n: int = N_QUESTIONS) -> float:
    # n questions: (tokens in x input price + tokens out x output price), prices per million tokens
    return n * (tokens_in * price_in + OUT_TOK * price_out) / 1e6


print(f"output tokens per answer (measured): {OUT_TOK}")
print("tokens in per question (measured): " + " | ".join(f"{name} {s['tokens']:,}" for name, s in designs.items()))
print("seconds per answer on this laptop: " + " | ".join(f"{name} {s['sec']:.2f}" for name, s in designs.items()))
print(f"\nUS$ for {N_QUESTIONS:,} questions = {N_QUESTIONS:,} x (tokens in x input price + tokens out x output price) / 1,000,000")
print(f"{'model':24} {'$/M in':>7} {'$/M out':>8}   " + "   ".join(f"{name:>16}" for name in designs))
for model, (pin, pout) in PRICES.items():
    print(f"{model:24} {pin:7.2f} {pout:8.2f}   " + "   ".join(f"{bill(s['tokens'], pin, pout):16.2f}" for s in designs.values()))
m, (pin, pout) = next(iter(PRICES.items()))
t_in = S_RAG["tokens"]
print(f"\nworked example, RAG on {m}: {t_in:,} x {N_QUESTIONS:,} = {t_in * N_QUESTIONS:,} tokens in x ${pin:.2f}/M = ${t_in * N_QUESTIONS * pin / 1e6:.2f}; "
      f"{OUT_TOK} x {N_QUESTIONS:,} = {OUT_TOK * N_QUESTIONS:,} tokens out x ${pout:.2f}/M = ${OUT_TOK * N_QUESTIONS * pout / 1e6:.2f}; "
      f"total ${bill(t_in, pin, pout):.2f} for {N_QUESTIONS:,} questions = ${bill(t_in, pin, pout) / N_QUESTIONS:.4f} a question")
print("token counts are this notebook's local model's; each provider counts tokens its own way, so these are estimates")

output tokens per answer (measured): 53
tokens in per question (measured): closed book 58 | RAG, top 5 568 | paste everything 10,529
seconds per answer on this laptop: closed book 1.54 | RAG, top 5 1.15 | paste everything 14.07

US$ for 1,000 questions = 1,000 x (tokens in x input price + tokens out x output price) / 1,000,000
model                     $/M in  $/M out        closed book         RAG, top 5   paste everything
claude-sonnet-5-5           2.00    10.00               0.65               1.67              21.59
gpt-6.1-sol                 2.00    10.00               0.65               1.67              21.59
gemini-3.1-pro-preview      2.00    12.00               0.75               1.77              21.69
gemini-3.8-flash            0.75     3.75               0.24               0.62               8.10
claude-haiku-4-5            1.00     5.00               0.32               0.83              10.79
gpt-4.1                     2.00     8.00               0.54               1.

---

# 6 — All the designs, compared

> **Lecture:** Block 6 · *All the designs, compared*

Every design from this notebook, scored on the same 35 questions. Read across a row before you read
down a column: quality and cost pull in different directions, and a rule that sounds safe has to earn
its place in the numbers. That is why a RAG system is a set of decisions rather than a library call.

- **right** — answerable questions whose answer passes its key (of 27).
- **leaked** — internal facts given out *from an internal page that was in the window* (of 5). (A model
  can also *guess* an internal number; that is a made-up answer, not a leak, and §4.3 deals with it.)
- **old policy** — stale questions whose answer states the superseded policy (of 3), whether it came
  from the old page or from the model's weights.
- **declined** — questions that should be declined and were (of 8).

In [ ]:
designs = [("closed book", closed), ("whole help centre pasted", everything), ("RAG, every page indexed", rag),
           ("RAG, filtered", filtered_only), ("RAG, filtered + answer checked", final)]
card = []
for name, df in designs:
    s = summary(df)
    card.append({"design": name, "right": f"{s['correct']}/{s['of']}",
                 "leaked": f"{s.get('leaked', 0)}/{s.get('restricted_of', 5)}",
                 "old policy": f"{s.get('stale', 0)}/{s.get('stale_of', 3)}",
                 "declined": f"{s.get('declined', 0)}/{s.get('decline_of', 8)}",
                 "tokens/call": f"{s['tokens']:,}", "s/call": f"{s['sec']:.2f}"})
SCORECARD = pd.DataFrame(card).set_index("design")
print(SCORECARD.to_string())

                                right leaked old policy declined tokens/call s/call
design                                                                             
closed book                      2/27    0/5        1/3      0/8          58   1.54
whole help centre pasted        19/27    5/5        1/3      1/8      10,529  14.07
RAG, every page indexed         18/27    5/5        3/3      1/8         568   1.15
RAG, filtered                   20/27    0/5        1/3      4/8         589   1.19
RAG, filtered + answer checked  19/27    0/5        1/3      4/8         589   1.17


---

## Where this goes next

- **Week 8 — agents and tools.** Today the application searched on every question. An agent decides
  *when* to search, and retrieval becomes one tool among several. The order lookup of §5.5 becomes
  one too: today the application ran the query itself, whenever it saw an order number; next week the
  model asks for the lookup, through a tool call.
- **Week 10 — evaluation and governance.** The golden questions and the two numbers (search, answer)
  become a release gate; today's filters are the first controls.

## Reference — what was not covered

- **Approximate search, in more detail.** Vector databases avoid scoring every vector: HNSW builds a graph of near
  neighbours and walks it; IVF groups vectors into clusters and searches only the nearest few. Both
  trade a little recall for a lot of speed at millions of passages (the lecture notes link to the Faiss guide).
- **Contextual retrieval.** Before embedding, a model writes one sentence of context onto every passage
  ("this is from the return policy, the fees section") — a repair for passages cut away from their
  meaning (Anthropic's post is linked in the lecture notes).
- **Query rewriting.** A model rewrites the customer's question into the help centre's vocabulary
  before searching, or writes a hypothetical answer and searches with that (HyDE).
- **Other stores.** A knowledge graph of entities and their relationships is retrieval too, like the
  SQL lookup of §5.5: anything that puts the right facts in the window.